# Eksperimen Komparatif NER Bahasa Indonesia: NERGRIT Gold vs Korpus Sintetis Nemotron 18-Kelas Baru (Terverifikasi Tahap 2)

**Cara Menjalankan di Google Colab (1-Klik):**
1. Buka notebook ini di Google Colab.
2. Pilih menu **Runtime → Change runtime type → Hardware accelerator: GPU (A100 / L4 / T4)**.
3. Klik **Runtime → Run all** dan izinkan akses **Google Drive** saat diminta.

---

### 1. Tujuan Eksperimen
Notebook mandiri (*self-contained*) ini dirancang untuk melatih dan mengevaluasi secara *head-to-head* model fondasi **`indobenchmark/indobert-large-p1`** (340M parameter, 24 layer) pada dua kondisi korpus latih:
1. **Skenario 1 — NERGRIT Gold Baseline (`gold`)**:
   Korpus anotasi manusia asli (`12.465` kalimat latih, `2.543` kalimat validasi) dengan normalisasi `REG` & `TTL` $\rightarrow$ `O` (**18 kelas entitas aktif / 37 tag BIO**).
2. **Skenario 2 — Korpus Sintetis Nemotron 18-Kelas Baru (`nemotron_18class`)**:
   Korpus hasil distilasi LLM yang telah selesai melalui **Verifikasi & Quality Control Tahap 2 (100% selesai: `17.525` kalimat, `119.680` entitas JSON / `118.596` rentang BIO)** menggunakan spesifikasi taksonomi 18 kelas ketat (`0` `REG`, `0` `TTL`, `0` kegagalan substring), disaring dengan penapis anti-kebocoran *character 5-gram Jaccard* $\ge 0,90$ terhadap Gold Test Set, dan dibagi secara deterministik ke dalam **85% Train (`14.896` kalimat) / 15% Validation (`2.629` kalimat)**.

Kedua model dievaluasi secara *blind* pada himpunan uji acuan manusia yang sama: **NERGRIT Gold Test Set (`2.423` kalimat, `8.890` rentang entitas)** menggunakan metrik **Exact Span Match tingkat CoNLL**.

---

### 2. Spesifikasi Hiperparameter Bersama (Identik 100% dengan Protokol Skripsi Tabel 3.2 & Tabel 4.13)

| Parameter Pelatihan | Nilai Konfigurasi | Keterangan |
| :--- | :--- | :--- |
| **Model Fondasi (*Backbone*)** | `indobenchmark/indobert-large-p1` | 24 layer, hidden size 1024, 340M parameter |
| **Taksonomi Label** | 18 Kelas Entitas / 37 Tag BIO | Eksklusi penuh `REG` & `TTL` $\rightarrow$ `O` |
| **Random Seed Pelatihan (`TRAINING_SEEDS`)** | `[42]` (1 Seed Deterministik) | Seed pembagian partisi `SPLIT_SEED = 42` |
| **Panjang Sekuens Maksimum (`max_seq_length`)** | `256` subword | *Lossless word-aligned chunking* & *recombination* |
| **Ukuran Batch Latih (`train_batch_size`)** | `32` (`grad_accum = 1`) | Ukuran batch efektif = 32 sekuens |
| **Ukuran Batch Evaluasi (`eval_batch_size`)** | `64` | Evaluasi akhir tiap epoch |
| **Laju Pembelajaran (`learning_rate`)** | `2e-5` ($2 \times 10^{-5}$) | Titik Pareto-optimal hasil penalaan empiris |
| **Pengoptimal (`optim`)** | `adamw_torch` | $\beta_1 = 0,9$; $\beta_2 = 0,999$; $\epsilon = 10^{-8}$; `weight_decay = 0,01` |
| **Penjadwal LR (`lr_scheduler_type`)** | `linear` (`warmup_ratio = 0,10`) | Pemanasan linier 10% dari total langkah |
| **Batas Epoch & Penghentian Dini** | `10` Epoch (`patience = 2`) | Pemantauan `eval_macro_f1` terbaik (`load_best_model_at_end = True`) |
| **Regularisasi & Gradien** | `dropout = 0,10`, `max_grad_norm = 1,0` | `pad_to_multiple_of = 8` |
| **Presisi Komputasi** | `bf16=True` (A100/L4) / `fp16=True` (T4) | Bobot utama model dalam `float32` |

---

### 3. Pencadangan & Pengunggahan Otomatis ke Google Drive
Seluruh tahapan eksperimen terintegrasi otomatis dengan **Google Drive** pada direktori:
`/content/drive/MyDrive/SKRIPSI_NER_BAB4/eksperimen_nemotron_18class/`
- **Sel 4**: Jika bundel input `nemotron_18class_colab_bundle.zip` diunduh dari GitHub, bundel tersebut langsung disalin otomatis ke Google Drive agar tersimpan permanen.
- **Sel 7 & Sel 8**: Setiap kali pelatihan model selesai, *checkpoint* model terbaik (`best_model`), riwayat pelatihan per epoch (`training_history.csv`/`.json`), dan manifes langsung dicadangkan otomatis ke Google Drive (mendukung *auto-resume* jika sesi Colab terputus).
- **Sel 10**: Seluruh tabel komparasi, prediksi `.jsonl`/`.conll`, analisis galat otomatis, dan grafik `.png` dikemas ke dalam **`nemotron_18class_experiment_results.zip`**, diunggah otomatis ke Google Drive, serta diunduh ke peramban Anda.

In [ ]:
# =============================================================================
# SEL 2: KONFIGURASI EKSPERIMEN, SEED [42], & MOUNT GOOGLE DRIVE OTOMATIS
# =============================================================================
from __future__ import annotations

from copy import deepcopy
import hashlib
import json
import os
from pathlib import Path
import shutil
import sys
import time

# 1. Taksonomi 18 Kelas Entitas Aktif (Tanpa REG dan TTL)
ACTIVE_TYPES = (
    "CRD", "DAT", "EVT", "FAC", "GPE", "LAN", "LAW", "LOC", "MON",
    "NOR", "ORD", "ORG", "PER", "PRC", "PRD", "QTY", "TIM", "WOA",
)
EXCLUDED_TYPES = ("REG", "TTL")
ALL_KNOWN_TYPES = frozenset(ACTIVE_TYPES) | frozenset(EXCLUDED_TYPES)
LABELS = ["O"] + [f"{prefix}-{kind}" for kind in ACTIVE_TYPES for prefix in ("B", "I")]
LABEL2ID = {tag: idx for idx, tag in enumerate(LABELS)}
ID2LABEL = {idx: tag for idx, tag in enumerate(LABELS)}

assert len(ACTIVE_TYPES) == 18, f"Harus tepat 18 kelas aktif, ditemukan: {len(ACTIVE_TYPES)}"
assert len(LABELS) == 37, f"Harus tepat 37 tag BIO, ditemukan: {len(LABELS)}"
assert not any("REG" in tag or "TTL" in tag for tag in LABELS), "REG dan TTL dilarang ada di LABELS!"

# 2. Konfigurasi Seed & Hiperparameter (100% Identik dengan Skripsi Tabel 3.2 & Tabel 4.13)
# Sesuai instruksi: gunakan 1 random seed (42) untuk NERGRIT Gold & Nemotron 18-Kelas Baru
TRAINING_SEEDS = [42]
SPLIT_SEED = 42
NEAR_THRESHOLD = 0.90

MODEL_CHECKPOINT = "indobenchmark/indobert-large-p1"
MAX_SEQ_LENGTH = 256
TRAIN_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 1
EVAL_BATCH_SIZE = 64
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.999
ADAM_EPSILON = 1e-8
WARMUP_RATIO = 0.10
LR_SCHEDULER_TYPE = "linear"
NUM_TRAIN_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_THRESHOLD = 0.0
METRIC_FOR_BEST_MODEL = "macro_f1"
LOAD_BEST_MODEL_AT_END = True
DROPOUT = 0.10
MAX_GRAD_NORM = 1.0
PAD_TO_MULTIPLE_OF = 8
OPTIM = "adamw_torch"
DATALOADER_NUM_WORKERS = 2
SAVE_BEST_WEIGHTS_TO_DRIVE = True

CONFIG = {
    "experiment_name": "eksperimen_komparatif_gold_vs_nemotron_18class",
    "model_checkpoint": MODEL_CHECKPOINT,
    "model_revision": None,
    "training_seeds": list(TRAINING_SEEDS),
    "split_seed": SPLIT_SEED,
    "near_threshold": NEAR_THRESHOLD,
    "active_types": list(ACTIVE_TYPES),
    "excluded_types": list(EXCLUDED_TYPES),
    "labels": list(LABELS),
    "max_seq_length": MAX_SEQ_LENGTH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "adam_beta1": ADAM_BETA1,
    "adam_beta2": ADAM_BETA2,
    "adam_epsilon": ADAM_EPSILON,
    "warmup_ratio": WARMUP_RATIO,
    "lr_scheduler_type": LR_SCHEDULER_TYPE,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_threshold": EARLY_STOPPING_THRESHOLD,
    "metric_for_best_model": METRIC_FOR_BEST_MODEL,
    "load_best_model_at_end": LOAD_BEST_MODEL_AT_END,
    "dropout": DROPOUT,
    "max_grad_norm": MAX_GRAD_NORM,
    "pad_to_multiple_of": PAD_TO_MULTIPLE_OF,
    "optim": OPTIM,
    "dataloader_num_workers": DATALOADER_NUM_WORKERS,
    "bf16": True,
    "fp16": False,
    "tf32": False,
    "gradient_checkpointing": False,
    "save_best_weights_to_drive": SAVE_BEST_WEIGHTS_TO_DRIVE,
}


def compute_sha256(path: Path) -> str:
    """Hitung SHA-256 berkas secara streaming."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def stable_json_hash(obj) -> str:
    """Hitung SHA-256 deterministik dari struktur data JSON."""
    payload = json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


# 3. Mount Otomatis Google Drive & Penyiapan Direktori Kerja
IN_COLAB = "google.colab" in sys.modules or Path("/content").is_dir()
DRIVE_MOUNTED = False

if "NEMOTRON_DRIVE_DIR" in os.environ:
    DRIVE_BASE_DIR = Path(os.environ["NEMOTRON_DRIVE_DIR"]).resolve()
    DRIVE_BASE_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_MOUNTED = True
elif IN_COLAB:
    try:
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").is_dir():
            print("🔗 Melakukan mount otomatis Google Drive (/content/drive)...")
            drive.mount("/content/drive", force_remount=False)
        DRIVE_BASE_DIR = Path("/content/drive/MyDrive/SKRIPSI_NER_BAB4/eksperimen_nemotron_18class")
        DRIVE_BASE_DIR.mkdir(parents=True, exist_ok=True)
        DRIVE_MOUNTED = True
        print(f"✅ Google Drive berhasil terhubung: {DRIVE_BASE_DIR}")
    except Exception as exc:
        print(f"⚠️ Peringatan: Gagal mount Google Drive otomatis ({exc}). Menggunakan fallback direktori lokal.")
        DRIVE_BASE_DIR = Path("/content/drive_fallback/eksperimen_nemotron_18class")
        DRIVE_BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    DRIVE_BASE_DIR = (Path.cwd() / "outputs" / "drive_mirror_nemotron_18class").resolve()
    DRIVE_BASE_DIR.mkdir(parents=True, exist_ok=True)

if "NEMOTRON_LOCAL_WORK_DIR" in os.environ:
    LOCAL_WORK_DIR = Path(os.environ["NEMOTRON_LOCAL_WORK_DIR"]).resolve()
elif IN_COLAB:
    LOCAL_WORK_DIR = Path("/content/nemotron_18class_workspace")
else:
    LOCAL_WORK_DIR = (Path.cwd() / "outputs" / "nemotron_18class_workspace").resolve()

DATA_DIR = LOCAL_WORK_DIR / "data"
PREPARED_DIR = LOCAL_WORK_DIR / "prepared"
OUTPUT_DIR = LOCAL_WORK_DIR / "outputs"
RUNS_DIR = OUTPUT_DIR / "runs"
TABLES_DIR = OUTPUT_DIR / "tables"
HISTORIES_DIR = OUTPUT_DIR / "histories"
PREDICTIONS_DIR = OUTPUT_DIR / "predictions"
ERROR_ANALYSIS_DIR = OUTPUT_DIR / "error_analysis"
FIGURES_DIR = OUTPUT_DIR / "figures"

for folder in (
    LOCAL_WORK_DIR, DATA_DIR, PREPARED_DIR, OUTPUT_DIR,
    RUNS_DIR, TABLES_DIR, HISTORIES_DIR, PREDICTIONS_DIR,
    ERROR_ANALYSIS_DIR, FIGURES_DIR,
    DRIVE_BASE_DIR, DRIVE_BASE_DIR / "runs", DRIVE_BASE_DIR / "tables",
    DRIVE_BASE_DIR / "histories", DRIVE_BASE_DIR / "predictions",
    DRIVE_BASE_DIR / "error_analysis", DRIVE_BASE_DIR / "figures",
):
    folder.mkdir(parents=True, exist_ok=True)

CONFIG["local_work_dir"] = str(LOCAL_WORK_DIR)
CONFIG["output_dir"] = str(OUTPUT_DIR)
CONFIG["drive_base_dir"] = str(DRIVE_BASE_DIR)
CONFIG["drive_mounted"] = DRIVE_MOUNTED


def sync_to_drive(src_path: Path | str, rel_dest: str | Path | None = None, verify_hash: bool = True) -> Path | None:
    """Salin otomatis berkas atau direktori ke Google Drive dan verifikasi integritasnya."""
    src = Path(src_path).resolve()
    if not src.exists():
        print(f"⚠️ [Drive Sync] Sumber tidak ditemukan: {src}")
        return None
    if rel_dest is None:
        try:
            rel_dest = src.relative_to(OUTPUT_DIR)
        except ValueError:
            rel_dest = src.name
    dest = (DRIVE_BASE_DIR / rel_dest).resolve()
    try:
        if src == dest:
            return dest
        if src.is_dir():
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(src, dest, dirs_exist_ok=True)
            print(f"☁️ [Drive Sync] Direktori tersinkronisasi ke Google Drive: {dest}")
        else:
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dest)
            if verify_hash:
                src_hash = compute_sha256(src)
                dst_hash = compute_sha256(dest)
                if src_hash != dst_hash:
                    raise RuntimeError(f"Ketidakcocokan SHA-256 saat menyalin {src.name} ke Google Drive!")
            size_mb = dest.stat().st_size / (1024 * 1024)
            print(f"☁️ [Drive Sync] Berkas tersimpan di Google Drive: {dest} ({size_mb:.2f} MB)")
        return dest
    except Exception as exc:
        print(f"⚠️ [Drive Sync] Gagal menyalin {src} ke {dest}: {exc}")
        return None


# Simpan konfigurasi awal dan sinkronkan ke Google Drive
config_path = OUTPUT_DIR / "experiment_config.json"
config_path.write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding="utf-8")
sync_to_drive(config_path, "experiment_config.json")

print("=" * 72)
print("⚙️ KONFIGURASI EKSPERIMEN SIAP")
print(f"   Model Fondasi        : {MODEL_CHECKPOINT}")
print(f"   Random Seed Latih    : {TRAINING_SEEDS} (1 seed deterministik)")
print(f"   Jumlah Kelas / Tag   : {len(ACTIVE_TYPES)} kelas aktif / {len(LABELS)} tag BIO")
print(f"   Direktori Kerja NVMe : {LOCAL_WORK_DIR}")
print(f"   Direktori Google Drive: {DRIVE_BASE_DIR} (Mounted={DRIVE_MOUNTED})")
print("=" * 72)

In [ ]:
# =============================================================================
# SEL 3: INSTALASI DEPENDENSI, PEMERIKSAAN GPU (BF16/FP16), & REPRODUKSIBILITAS
# =============================================================================
import importlib
import importlib.util
import platform
import random
import subprocess

# 1. Kebijakan Determinisme CUDA (diatur sebelum inisialisasi konteks kernel GPU)
os.environ.pop("CUDA_LAUNCH_BLOCKING", None)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["FLASH_ATTENTION_DETERMINISTIC"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("TQDM_MININTERVAL", "2")

# 2. Pemeriksaan & Pemasangan Pustaka Tanpa Merusak PyTorch CUDA Bawaan Colab
REQUIRED_PACKAGES = [
    ("transformers", "transformers>=4.40.0"),
    ("datasets", "datasets>=2.18.0"),
    ("accelerate", "accelerate>=0.28.0"),
    ("seqeval", "seqeval>=1.2.2"),
    ("evaluate", "evaluate>=0.4.1"),
    ("sklearn", "scikit-learn>=1.3.0"),
    ("matplotlib", "matplotlib>=3.7.0"),
]

missing_specs = [
    pip_spec
    for module_name, pip_spec in REQUIRED_PACKAGES
    if importlib.util.find_spec(module_name) is None
]

if missing_specs and (IN_COLAB or os.environ.get("NEMOTRON_FORCE_PIP_INSTALL") == "1") and os.environ.get("NEMOTRON_SKIP_PIP") != "1":
    print(f"📦 Memasang pustaka yang belum tersedia: {missing_specs} ...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--disable-pip-version-check", *missing_specs
    ])
    print("✅ Pemasangan pustaka selesai.")
elif not missing_specs:
    print("✅ Seluruh pustaka utama sudah terpasang di lingkungan runtime.")
else:
    print(f"ℹ️ Mode uji lokal tanpa instalasi pip otomatis (pustaka opsional belum terpasang: {missing_specs}).")

TORCH_AVAILABLE = importlib.util.find_spec("torch") is not None
TRANSFORMERS_AVAILABLE = importlib.util.find_spec("transformers") is not None

if TORCH_AVAILABLE:
    import torch
else:
    torch = None  # type: ignore

if TRANSFORMERS_AVAILABLE:
    import transformers
    from transformers import set_seed
else:
    transformers = None  # type: ignore
    set_seed = None  # type: ignore


def configure_reproducibility(seed: int = 42) -> None:
    """Atur seluruh generator angka acak dan bendera determinisme kernel."""
    os.environ.pop("CUDA_LAUNCH_BLOCKING", None)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    os.environ["FLASH_ATTENTION_DETERMINISTIC"] = "1"
    random.seed(seed)
    if set_seed is not None:
        set_seed(seed, deterministic=True)
    if torch is not None:
        torch.manual_seed(seed)
        if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cuda.matmul.allow_tf32 = False
            torch.backends.cudnn.allow_tf32 = False


configure_reproducibility(TRAINING_SEEDS[0])

# 3. Deteksi Akselerator GPU & Penyesuaian Presisi Campuran (BF16 di A100/L4, FP16 di T4)
CUDA_AVAILABLE = bool(torch is not None and torch.cuda.is_available())
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "CPU"
GPU_COUNT = torch.cuda.device_count() if CUDA_AVAILABLE else 0
BF16_SUPPORTED = bool(CUDA_AVAILABLE and torch.cuda.is_bf16_supported())

if CUDA_AVAILABLE and BF16_SUPPORTED:
    USE_BF16 = True
    USE_FP16 = False
    PRECISION_MODE = "bf16"
elif CUDA_AVAILABLE:
    USE_BF16 = False
    USE_FP16 = True
    PRECISION_MODE = "fp16"
else:
    USE_BF16 = False
    USE_FP16 = False
    PRECISION_MODE = "fp32-cpu"

CONFIG["bf16"] = USE_BF16
CONFIG["fp16"] = USE_FP16
CONFIG["precision_mode"] = PRECISION_MODE

VRAM_GIB = (
    round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2)
    if CUDA_AVAILABLE else 0.0
)
COMPUTE_CAPABILITY = (
    list(torch.cuda.get_device_capability(0))
    if CUDA_AVAILABLE else None
)

ENV_FINGERPRINT = {
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "torch_version": getattr(torch, "__version__", None),
    "torch_cuda_version": getattr(getattr(torch, "version", None), "cuda", None),
    "cudnn_version": (torch.backends.cudnn.version() if CUDA_AVAILABLE else None),
    "transformers_version": getattr(transformers, "__version__", None),
    "cuda_available": CUDA_AVAILABLE,
    "gpu_name": GPU_NAME,
    "gpu_count": GPU_COUNT,
    "vram_gib": VRAM_GIB,
    "compute_capability": COMPUTE_CAPABILITY,
    "bf16_supported": BF16_SUPPORTED,
    "precision_mode": PRECISION_MODE,
    "deterministic_algorithms": (
        torch.are_deterministic_algorithms_enabled() if torch is not None else True
    ),
    "cudnn_deterministic": (
        torch.backends.cudnn.deterministic if torch is not None else True
    ),
    "cudnn_benchmark": (
        torch.backends.cudnn.benchmark if torch is not None else False
    ),
    "tf32_allowed": False,
}

fingerprint_path = OUTPUT_DIR / "environment_fingerprint.json"
fingerprint_path.write_text(json.dumps(ENV_FINGERPRINT, indent=2, ensure_ascii=False), encoding="utf-8")
(OUTPUT_DIR / "experiment_config.json").write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding="utf-8")
sync_to_drive(fingerprint_path, "environment_fingerprint.json")
sync_to_drive(OUTPUT_DIR / "experiment_config.json", "experiment_config.json")

print("=" * 72)
print("🖥️ LINGKUNGAN KOMPUTASI & AKSELERATOR GPU")
print(f"   Perangkat GPU       : {GPU_NAME} ({VRAM_GIB:.2f} GiB VRAM)")
print(f"   Compute Capability  : {COMPUTE_CAPABILITY}")
print(f"   Mode Presisi Aktif  : {PRECISION_MODE.upper()} (bf16={USE_BF16}, fp16={USE_FP16})")
print(f"   PyTorch / CUDA      : {ENV_FINGERPRINT['torch_version']} / CUDA {ENV_FINGERPRINT['torch_cuda_version']}")
print(f"   Transformers        : {ENV_FINGERPRINT['transformers_version']}")
print("=" * 72)

In [ ]:
# =============================================================================
# SEL 4: UNDUH / MUAT BUNDLE DATA, UPLOAD OTOMATIS KE GOOGLE DRIVE, & VERIFIKASI SHA-256
# =============================================================================
import urllib.request
import zipfile

BUNDLE_FILENAME = "nemotron_18class_colab_bundle.zip"
GITHUB_BUNDLE_URLS = (
    "https://raw.githubusercontent.com/Treamyracle/bab4/main/notebooks/nemotron_18class_colab_bundle.zip",
    "https://raw.githubusercontent.com/Treamyracle/skripsi/main/notebooks/nemotron_18class_colab_bundle.zip",
    "https://github.com/Treamyracle/bab4/raw/main/notebooks/nemotron_18class_colab_bundle.zip",
    "https://github.com/Treamyracle/skripsi/raw/main/notebooks/nemotron_18class_colab_bundle.zip",
)

REQUIRED_BUNDLE_MEMBERS = (
    "gold/train.txt",
    "gold/valid.txt",
    "gold/test.txt",
    "gold/labels-map.csv",
    "nemotron_18class/corrected.jsonl",
    "bundle_manifest.json",
)


def locate_or_download_bundle(
    local_work_dir: Path = LOCAL_WORK_DIR,
    drive_base_dir: Path = DRIVE_BASE_DIR,
    urls: tuple[str, ...] = GITHUB_BUNDLE_URLS,
) -> tuple[Path, str]:
    """Cari bundel data di lokal/Drive atau unduh otomatis dari GitHub, lalu unggah otomatis ke Google Drive."""
    drive_bundle_path = drive_base_dir / BUNDLE_FILENAME
    local_candidates = [
        Path.cwd() / "notebooks" / BUNDLE_FILENAME,
        Path.cwd() / BUNDLE_FILENAME,
        Path("/content") / BUNDLE_FILENAME,
        local_work_dir / BUNDLE_FILENAME,
        drive_bundle_path,
        Path("/content/drive/MyDrive/SKRIPSI_NER_BAB4") / BUNDLE_FILENAME,
    ]

    found_path: Path | None = None
    source_origin = "unknown"

    for candidate in local_candidates:
        if candidate.is_file() and candidate.stat().st_size > 100_000:
            found_path = candidate.resolve()
            source_origin = "google_drive" if str(drive_base_dir) in str(found_path) else "local_filesystem"
            print(f"📂 Bundel data ditemukan ({source_origin}): {found_path} ({found_path.stat().st_size / (1024 * 1024):.2f} MB)")
            break

    if found_path is None:
        download_target = local_work_dir / BUNDLE_FILENAME
        download_target.parent.mkdir(parents=True, exist_ok=True)
        last_error: Exception | None = None
        for url in urls:
            try:
                print(f"🌐 Mengunduh {BUNDLE_FILENAME} dari GitHub:\n   {url}")
                req = urllib.request.Request(
                    url,
                    headers={"User-Agent": "IndoBERT-Nemotron18Class-Colab/1.0"},
                )
                t0 = time.time()
                with urllib.request.urlopen(req, timeout=120) as resp, download_target.open("wb") as out_f:
                    shutil.copyfileobj(resp, out_f)
                if download_target.stat().st_size < 100_000:
                    raise RuntimeError("Ukuran berkas unduhan terlalu kecil / tidak valid.")
                elapsed = time.time() - t0
                print(f"✅ Unduhan dari GitHub selesai dalam {elapsed:.2f}s ({download_target.stat().st_size / (1024 * 1024):.2f} MB).")
                found_path = download_target.resolve()
                source_origin = f"github:{url}"
                break
            except Exception as exc:
                last_error = exc
                print(f"⚠️ Gagal mengunduh dari {url}: {exc}")

        if found_path is None and IN_COLAB:
            try:
                from google.colab import files  # type: ignore
                print("📤 Silakan unggah berkas 'nemotron_18class_colab_bundle.zip' secara manual:")
                uploaded = files.upload()
                if uploaded:
                    first_name = next(iter(uploaded))
                    download_target.write_bytes(uploaded[first_name])
                    found_path = download_target.resolve()
                    source_origin = "colab_manual_upload"
            except Exception as exc:
                last_error = exc

        if found_path is None:
            raise FileNotFoundError(
                f"Tidak dapat menemukan atau mengunduh {BUNDLE_FILENAME}. Galat terakhir: {last_error}"
            )

    # Pastikan bundel otomatis disalin/diunggah ke Google Drive agar tersimpan permanen
    try:
        need_drive_copy = (
            not drive_bundle_path.is_file()
            or compute_sha256(drive_bundle_path) != compute_sha256(found_path)
        )
        if need_drive_copy:
            print(f"☁️ Menyimpan otomatis {BUNDLE_FILENAME} ke Google Drive ({drive_bundle_path})...")
            sync_to_drive(found_path, BUNDLE_FILENAME, verify_hash=True)
        else:
            print(f"☁️ Salinan {BUNDLE_FILENAME} di Google Drive sudah mutakhir dan terverifikasi SHA-256.")
    except Exception as exc:
        print(f"⚠️ Peringatan: Gagal menyinkronkan bundel ke Google Drive: {exc}")

    return found_path, source_origin


def extract_and_verify_bundle(bundle_path: Path, extract_dir: Path = DATA_DIR) -> dict:
    """Ekstrak bundel input dan verifikasi integritas SHA-256 serta jumlah kalimat."""
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_path, "r") as zf:
        names = set(zf.namelist())
        missing = [req for req in REQUIRED_BUNDLE_MEMBERS if req not in names]
        if missing:
            raise ValueError(f"Bundel data tidak lengkap! Berkas hilang: {missing}")
        zf.extractall(extract_dir)

    manifest_path = extract_dir / "bundle_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected_hashes = manifest.get("sha256", {})

    verified_files = {}
    for rel_path, expected_sha in expected_hashes.items():
        target = extract_dir / rel_path
        if not target.is_file():
            raise FileNotFoundError(f"Berkas hasil ekstraksi tidak ditemukan: {target}")
        actual_sha = compute_sha256(target)
        if actual_sha != expected_sha:
            raise ValueError(
                f"Verifikasi SHA-256 gagal untuk {rel_path}: diharapkan {expected_sha}, aktual {actual_sha}"
            )
        verified_files[rel_path] = actual_sha

    nemotron_jsonl = extract_dir / "nemotron_18class" / "corrected.jsonl"
    nemotron_lines = [
        line for line in nemotron_jsonl.read_text(encoding="utf-8").splitlines() if line.strip()
    ]
    expected_nemotron_count = manifest.get("record_counts", {}).get("nemotron_18class_sentences", 17525)
    if len(nemotron_lines) != expected_nemotron_count:
        raise ValueError(
            f"Jumlah kalimat Nemotron 18-Kelas tidak cocok: diharapkan {expected_nemotron_count}, ditemukan {len(nemotron_lines)}"
        )

    bundle_report = {
        "bundle_path": str(bundle_path),
        "bundle_sha256": compute_sha256(bundle_path),
        "extracted_dir": str(extract_dir),
        "verified_files": verified_files,
        "record_counts": manifest.get("record_counts", {}),
        "nemotron_sentences_verified": len(nemotron_lines),
    }
    report_path = OUTPUT_DIR / "bundle_verification.json"
    report_path.write_text(json.dumps(bundle_report, indent=2, ensure_ascii=False), encoding="utf-8")
    sync_to_drive(report_path, "bundle_verification.json")
    return bundle_report


BUNDLE_PATH, BUNDLE_ORIGIN = locate_or_download_bundle()
BUNDLE_VERIFICATION = extract_and_verify_bundle(BUNDLE_PATH, DATA_DIR)

print("=" * 72)
print("📦 BUNDEL DATA INPUT TERVERIFIKASI (SHA-256 LULUS 100%)")
print(f"   Sumber Bundel        : {BUNDLE_ORIGIN}")
print(f"   Lokasi Bundel Lokal  : {BUNDLE_PATH}")
print(f"   SHA-256 Bundel       : {BUNDLE_VERIFICATION['bundle_sha256']}")
print(f"   Kalimat Nemotron 18C : {BUNDLE_VERIFICATION['nemotron_sentences_verified']:,} kalimat")
for rel_file, sha_val in sorted(BUNDLE_VERIFICATION["verified_files"].items()):
    print(f"   ✓ {rel_file:<34} sha256={sha_val[:16]}...")
print("=" * 72)

In [ ]:
# =============================================================================
# SEL 5: PRE-PROCESSING, ANTI-LEAKAGE SPLIT (JACCARD 5-GRAM >= 0.90), & STATISTIK 18 KELAS
# =============================================================================
from collections import Counter, defaultdict
import csv
import math
import random
import re
import statistics
import unicodedata

WORD_PATTERN = re.compile(r"\w+|[^\s\w]", flags=re.UNICODE)
ALLOWED_TYPES_SET = frozenset(ACTIVE_TYPES)

TYPE_ALIASES = {
    "PRO": "PRD", "PROD": "PRD", "PRODUCT": "PRD", "TECH": "PRD",
    "PERSON": "PER", "LOCATION": "LOC", "ORGANIZATION": "ORG", "DATE": "DAT",
    "TIME": "TIM", "MONEY": "MON", "QUANTITY": "QTY", "CARDINAL": "CRD",
    "ORDINAL": "ORD", "PERCENT": "PRC", "PCT": "PRC", "EVENT": "EVT",
    "FACILITY": "FAC", "LANGUAGE": "LAN", "WORK_OF_ART": "WOA", "NORP": "NOR",
    "REGION": "REG", "TITLE": "TTL",
}
GOLD_EXTRA_ALIASES = {"O\u201a\u00c4\u00e9": "O", "O -": "O"}


def write_csv_rows(path: Path, rows: list[dict]) -> None:
    """Simpan daftar dictionary ke berkas CSV UTF-8-SIG."""
    rows = list(rows)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    fields = list(dict.fromkeys(k for row in rows for k in row))
    with path.open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def align_entities_to_bio(text: str, entities: list[dict[str, str]]) -> list[tuple[str, str]]:
    """Tokenisasi kalimat dan selaraskan rentang entitas ke tag BIO 18 kelas."""
    tokens = re.findall(r"\w+|[^\w\s]", text, re.UNICODE)
    tags = ["O"] * len(tokens)
    tagged = [False] * len(tokens)

    sorted_ents = sorted(entities, key=lambda x: len(x.get("entity", "")), reverse=True)
    for ent in sorted_ents:
        ent_str = ent.get("entity", "").strip()
        ent_type = ent.get("type", "").strip()
        if not ent_str or ent_type not in ALLOWED_TYPES_SET:
            continue
        ent_tokens = re.findall(r"\w+|[^\w\s]", ent_str, re.UNICODE)
        if not ent_tokens:
            continue
        n = len(ent_tokens)
        for i in range(len(tokens) - n + 1):
            if tokens[i:i + n] == ent_tokens and not any(tagged[i:i + n]):
                tags[i] = f"B-{ent_type}"
                tagged[i] = True
                for j in range(1, n):
                    tags[i + j] = f"I-{ent_type}"
                    tagged[i + j] = True
                break
    return list(zip(tokens, tags))


def entity_spans(tags: list[str]) -> list[tuple[int, int, str]]:
    """Ekstrak rentang entitas (start, end-exclusive, type) dari barisan tag BIO (konvensi CoNLL/seqeval)."""
    result = []
    start, kind = None, None
    for index, tag in enumerate(list(tags) + ["O"]):
        if tag == "O":
            prefix, current = "O", None
        elif isinstance(tag, str) and re.fullmatch(r"[BI]-.+", tag):
            prefix, current = tag.split("-", 1)
        else:
            raise ValueError(f"Tag BIO tidak valid pada indeks {index}: {tag!r}")
        if start is not None and (prefix != "I" or current != kind):
            result.append((start, index, kind))
            start, kind = None, None
        if prefix != "O" and start is None:
            start, kind = index, current
    return result


def _percentile(values: list[int | float], q: float) -> float:
    if not values:
        return 0.0
    ordered = sorted(values)
    position = (len(ordered) - 1) * q
    lower = int(position)
    upper = min(lower + 1, len(ordered) - 1)
    return ordered[lower] + (ordered[upper] - ordered[lower]) * (position - lower)


def dataset_statistics(records: list[dict]) -> dict:
    """Hitung statistik deskriptif korpus: kalimat, token, entitas, kepadatan, dan distribusi 18 kelas."""
    records = list(records)
    lengths, counts, vocabulary = [], [], set()
    class_counts, entity_tokens = Counter(), 0
    for record in records:
        tokens, tags = record["tokens"], record["tags"]
        if len(tokens) != len(tags):
            raise ValueError(f"Ketidakcocokan panjang token/tag pada {record.get('sentence_id', '?')}")
        spans_list = entity_spans(tags)
        lengths.append(len(tokens))
        counts.append(len(spans_list))
        vocabulary.update(unicodedata.normalize("NFKC", token).casefold() for token in tokens)
        class_counts.update(kind for _, _, kind in spans_list)
        entity_tokens += sum(end - start for start, end, _ in spans_list)
    sentences, tokens_total, entities_total = len(records), sum(lengths), sum(counts)
    return {
        "S": sentences,
        "T": tokens_total,
        "E": entities_total,
        "T_entity": entity_tokens,
        "avg_tokens": tokens_total / sentences if sentences else 0.0,
        "avg_entities": entities_total / sentences if sentences else 0.0,
        "density_token_pct": 100.0 * entity_tokens / tokens_total if tokens_total else 0.0,
        "entities_per_100_tokens": 100.0 * entities_total / tokens_total if tokens_total else 0.0,
        "class_counts": {kind: class_counts[kind] for kind in ACTIVE_TYPES},
        "class_proportions_pct": {
            kind: (100.0 * class_counts[kind] / entities_total if entities_total else 0.0)
            for kind in ACTIVE_TYPES
        },
        "unique_vocab": len(vocabulary),
        "median_tokens": statistics.median(lengths) if lengths else 0.0,
        "p95_tokens": _percentile(lengths, 0.95),
        "no_entity_sentences": sum(c == 0 for c in counts),
        "no_entity_pct": 100.0 * sum(c == 0 for c in counts) / sentences if sentences else 0.0,
    }


def normalized_text(record: dict) -> str:
    """Normalisasi teks kanonis untuk pemeriksaan duplikasi & kebocoran antar-partisi."""
    text = unicodedata.normalize("NFKC", " ".join(record["tokens"]))
    return " ".join(WORD_PATTERN.findall(text)).casefold()


def _read_aliases(path: Path) -> dict[str, str]:
    aliases = {}
    for line in path.read_text(encoding="utf-8-sig").splitlines():
        if not line.strip():
            continue
        values = line.split()
        if len(values) != 2:
            raise ValueError(f"Baris alias label tidak valid di {path}: {line!r}")
        aliases[values[0]] = values[1]
    return aliases


def _normalize_tag(raw: str, aliases: dict[str, str]) -> str:
    tag = aliases.get(raw, raw).strip().upper()
    if tag == "O":
        return tag
    if not re.fullmatch(r"[BI]-.+", tag):
        raise ValueError(f"Label tidak dikenali: {raw!r}")
    prefix, kind = tag.split("-", 1)
    kind = TYPE_ALIASES.get(kind, kind)
    if kind not in ALL_KNOWN_TYPES:
        raise ValueError(f"Tipe entitas tidak dikenal pada label {raw!r}")
    return f"{prefix}-{kind}"


def build_record(dataset: str, split: str, source_ref: str, tokens: list[str], raw_tags: list[str], aliases: dict[str, str]) -> dict:
    """Bangun satu record standar dengan pemetaan REG & TTL -> O."""
    if not tokens or len(tokens) != len(raw_tags):
        raise ValueError(f"Kalimat kosong atau panjang token/tag berbeda di {source_ref}")
    original_tags, changes, invalid_i = [], [], []
    for index, (token, raw_tag) in enumerate(zip(tokens, raw_tags)):
        tag = _normalize_tag(raw_tag, aliases)
        if tag != raw_tag:
            changes.append({"kind": "label_alias", "source_token": index, "from": raw_tag, "to": tag})
        if tag.startswith("I-") and (
            not original_tags or original_tags[-1] not in ("B-" + tag[2:], "I-" + tag[2:])
        ):
            invalid_i.append(index)
        original_tags.append(tag)
    tags = ["O" if tag[2:] in ("REG", "TTL") else tag for tag in original_tags]
    sentence_id = dataset + "-" + stable_json_hash({"source": source_ref, "tokens": tokens, "tags": raw_tags})[:20]
    return {
        "sentence_id": sentence_id,
        "dataset": dataset,
        "source_split": split,
        "source_refs": [source_ref],
        "tokens": list(tokens),
        "original_tags": original_tags,
        "tags": tags,
        "group_id": "",
        "changes": changes,
        "invalid_I_positions": invalid_i,
    }


class _UnionFind:
    def __init__(self, count: int):
        self.parent = list(range(count))
        self.size = [1] * count

    def find(self, item: int) -> int:
        while item != self.parent[item]:
            self.parent[item] = self.parent[self.parent[item]]
            item = self.parent[item]
        return item

    def union(self, left: int, right: int) -> None:
        left, right = self.find(left), self.find(right)
        if left == right:
            return
        if self.size[left] < self.size[right]:
            left, right = right, left
        self.parent[right] = left
        self.size[left] += self.size[right]


def _grams(text: str) -> set[str]:
    return {text[index:index + 5] for index in range(max(0, len(text) - 4))}


def _similar_sorted(left: list[int], right: list[int], threshold: float) -> bool:
    required = math.ceil(threshold * (len(left) + len(right)) / (1 + threshold) - 1e-12)
    i = j = overlap = 0
    while i < len(left) and j < len(right):
        if overlap + min(len(left) - i, len(right) - j) < required:
            return False
        if left[i] == right[j]:
            overlap += 1
            i += 1
            j += 1
        elif left[i] < right[j]:
            i += 1
        else:
            j += 1
    return overlap >= required


def duplicate_groups(records: list[dict], threshold: float = NEAR_THRESHOLD) -> tuple[dict[str, str], dict]:
    """Pengelompokan duplikat eksak dan near-duplicate (character 5-gram Jaccard >= 0.90)."""
    records = list(records)
    texts, representatives = {}, []
    for index, record in enumerate(records):
        text = normalized_text(record)
        texts.setdefault(text, []).append(index)
    union = _UnionFind(len(records))
    for members in texts.values():
        for member in members[1:]:
            union.union(members[0], member)
        if len(normalized_text(records[members[0]]).split()) >= 10:
            representatives.append(members[0])
    gram_sets = {index: _grams(normalized_text(records[index])) for index in representatives}
    frequencies = Counter()
    for values in gram_sets.values():
        frequencies.update(values)
    order = {
        gram: rank
        for rank, gram in enumerate(sorted(frequencies, key=lambda gram: (frequencies[gram], gram)))
    }
    ranked = {index: sorted(order[gram] for gram in values) for index, values in gram_sets.items()}
    del gram_sets, frequencies, order
    inverted = defaultdict(list)
    comparisons = links = 0
    for index in sorted(representatives, key=lambda item: (len(ranked[item]), normalized_text(records[item]))):
        values = ranked[index]
        size = len(values)
        if not size:
            continue
        prefix_size = size - math.ceil(threshold * size - 1e-12) + 1
        minimum_size = math.ceil(threshold * size - 1e-12)
        candidates = set()
        for gram in values[:prefix_size]:
            for previous in inverted[gram]:
                if len(ranked[previous]) >= minimum_size:
                    candidates.add(previous)
        for previous in candidates:
            if union.find(previous) == union.find(index):
                continue
            comparisons += 1
            if _similar_sorted(values, ranked[previous], threshold):
                union.union(previous, index)
                links += 1
        for gram in values[:prefix_size]:
            inverted[gram].append(index)
    members_by_root = defaultdict(list)
    for index in range(len(records)):
        members_by_root[union.find(index)].append(index)
    mapping, multi = {}, []
    for members in members_by_root.values():
        group_id = "grp-" + stable_json_hash(sorted({normalized_text(records[idx]) for idx in members}))[:20]
        for index in members:
            mapping[records[index]["sentence_id"]] = group_id
        if len(members) > 1:
            multi.append({
                "group_id": group_id,
                "sentence_ids": [records[idx]["sentence_id"] for idx in members],
            })
    return mapping, {
        "method": "exact_character_5gram_Jaccard_prefix_join",
        "threshold": threshold,
        "minimum_tokens": 10,
        "candidate_comparisons": comparisons,
        "near_links": links,
        "multi_record_groups": len(multi),
    }


def assert_no_leakage(prepared: dict[str, dict[str, list[dict]]]) -> dict:
    """Pastikan tidak ada group_id yang muncul di lebih dari satu partisi (train/validation/test)."""
    roles = defaultdict(set)
    ids = set()
    for dataset_key, splits in prepared.items():
        for split_name, records in splits.items():
            for record in records:
                sid = record["sentence_id"]
                if sid in ids:
                    raise ValueError(f"Sentence ID duplikat terdeteksi: {sid}")
                ids.add(sid)
                roles[record["group_id"]].add(split_name)
    violating = {gid: sorted(sp) for gid, sp in roles.items() if len(sp) > 1}
    if violating:
        raise ValueError(f"Kebocoran lintas-partisi terdeteksi pada grup: {violating}")
    return {
        "passed": True,
        "cross_partition_overlap_groups": 0,
        "scope": "global normalized exact text and character-5gram Jaccard >= 0.90 for >= 10 tokens",
    }


def load_and_prepare_corpora(
    data_dir: Path = DATA_DIR,
    prepared_dir: Path = PREPARED_DIR,
    split_seed: int = SPLIT_SEED,
) -> tuple[dict[str, dict[str, list[dict]]], dict]:
    """Muat NERGRIT Gold & Nemotron 18-Kelas Baru, lakukan anti-leakage screening, dan bagi 85/15."""
    aliases = {**GOLD_EXTRA_ALIASES, **_read_aliases(data_dir / "gold" / "labels-map.csv")}
    gold_rows = []
    for split, filename in (("train", "train.txt"), ("validation", "valid.txt"), ("test", "test.txt")):
        rel_path = f"gold/{filename}"
        token_buf, tag_buf, sent_idx = [], [], 0
        lines = (data_dir / "gold" / filename).read_text(encoding="utf-8-sig").splitlines() + [""]
        for line in lines:
            if not line.strip():
                if token_buf:
                    gold_rows.append(
                        build_record("gold", split, f"{rel_path}#{sent_idx}", token_buf, tag_buf, aliases)
                    )
                    sent_idx += 1
                    token_buf, tag_buf = [], []
                continue
            content = line.rstrip()
            values = content.rsplit("\t", 1) if "\t" in content else content.rsplit(None, 1)
            if len(values) != 2:
                raise ValueError(f"Baris CoNLL tidak valid di {rel_path}: {line!r}")
            token_buf.append(values[0])
            tag_buf.append(values[1])

    nemotron_rows = []
    nemotron_jsonl_path = data_dir / "nemotron_18class" / "corrected.jsonl"
    for idx, line in enumerate(nemotron_jsonl_path.read_text(encoding="utf-8").splitlines()):
        if not line.strip():
            continue
        item = json.loads(line)
        text = item.get("text", "")
        entities = item.get("entities", [])
        for ent in entities:
            if ent.get("type") in ("REG", "TTL") or ent.get("type") not in ALLOWED_TYPES_SET:
                raise ValueError(f"Entitas terlarang di corrected.jsonl#{idx}: {ent}")
        bio_pairs = align_entities_to_bio(text, entities)
        if not bio_pairs:
            raise ValueError(f"Kalimat kosong setelah tokenisasi di corrected.jsonl#{idx}")
        toks = [w for w, _ in bio_pairs]
        tags = [t for _, t in bio_pairs]
        nemotron_rows.append(
            build_record("nemotron_18class", "unassigned", f"nemotron_18class/corrected.jsonl#{idx}", toks, tags, {})
        )

    flat = gold_rows + nemotron_rows
    groups, near_report = duplicate_groups(flat, threshold=NEAR_THRESHOLD)
    by_group = defaultdict(list)
    for record in flat:
        record["group_id"] = groups[record["sentence_id"]]
        by_group[record["group_id"]].append(record)

    group_role = {}
    for group_id, recs in by_group.items():
        fixed = {r["source_split"] for r in recs if r["dataset"] == "gold"}
        if "test" in fixed:
            group_role[group_id] = "test"
        elif "validation" in fixed:
            group_role[group_id] = "validation"
        elif "train" in fixed:
            group_role[group_id] = "train"

    # Anti-leakage: jika ada kalimat Nemotron yang satu grup dengan Gold Test, eksklusi dari train/validation
    leakage_excluded = [
        r["sentence_id"]
        for r in nemotron_rows
        if group_role.get(r["group_id"]) == "test"
    ]
    eligible_nemotron = sum(1 for r in nemotron_rows if group_role.get(r["group_id"]) != "test")
    target_val = round(eligible_nemotron * 0.15)
    current_val = sum(1 for r in nemotron_rows if group_role.get(r["group_id"]) == "validation")

    free_groups = sorted(gid for gid in by_group if gid not in group_role)
    random.Random(split_seed).shuffle(free_groups)
    for gid in free_groups:
        nem_count = sum(1 for r in by_group[gid] if r["dataset"] == "nemotron_18class")
        improvement = abs(target_val - current_val) - abs(target_val - current_val - nem_count)
        role = "validation" if improvement > 0 else "train"
        group_role[gid] = role
        if role == "validation":
            current_val += nem_count

    prepared = {
        "gold": {"train": [], "validation": [], "test": []},
        "nemotron_18class": {"train": [], "validation": []},
    }
    gold_moves = []
    for record in gold_rows:
        role = group_role[record["group_id"]]
        record["assigned_split"] = role
        if record["source_split"] != role:
            gold_moves.append({
                "sentence_id": record["sentence_id"],
                "from": record["source_split"],
                "to": role,
            })
        prepared["gold"][role].append(record)

    for record in nemotron_rows:
        role = group_role[record["group_id"]]
        if role == "test":
            continue
        record["assigned_split"] = role
        prepared["nemotron_18class"][role].append(record)

    for dataset_key, splits in prepared.items():
        for split_name, recs in splits.items():
            recs.sort(key=lambda r: r["sentence_id"])

    leakage_check = assert_no_leakage(prepared)

    # Simpan partisi JSONL dan hitung statistik
    prepared_dir.mkdir(parents=True, exist_ok=True)
    stats_by_dataset = {}
    split_hashes = {}
    for dataset_key, splits in prepared.items():
        stats_by_dataset[dataset_key] = {}
        split_hashes[dataset_key] = {}
        all_recs = []
        for split_name, recs in splits.items():
            out_jsonl = prepared_dir / dataset_key / f"{split_name}.jsonl"
            out_jsonl.parent.mkdir(parents=True, exist_ok=True)
            with out_jsonl.open("w", encoding="utf-8", newline="\n") as handle:
                for rec in recs:
                    handle.write(json.dumps(rec, ensure_ascii=False, sort_keys=True, separators=(",", ":")) + "\n")
            split_hashes[dataset_key][split_name] = compute_sha256(out_jsonl)
            stats_by_dataset[dataset_key][split_name] = dataset_statistics(recs)
            all_recs.extend(recs)
        stats_by_dataset[dataset_key]["all"] = dataset_statistics(all_recs)

    manifest = {
        "format": "nemotron-18class-prepared-v1",
        "split_seed": split_seed,
        "near_threshold": NEAR_THRESHOLD,
        "labels": LABELS,
        "active_types": list(ACTIVE_TYPES),
        "near_duplicate_report": near_report,
        "gold_split_moves_count": len(gold_moves),
        "nemotron_test_leakage_excluded_count": len(leakage_excluded),
        "leakage_check": leakage_check,
        "split_hashes": split_hashes,
        "statistics": stats_by_dataset,
    }
    manifest["prepared_hash"] = stable_json_hash({
        "split_hashes": split_hashes,
        "labels": LABELS,
        "split_seed": split_seed,
    })

    manifest_path = prepared_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

    # Ekspor tabel statistik korpus & distribusi 18 kelas ke CSV
    stat_rows = []
    for d_key, d_label in (("gold", "NERGRIT Gold"), ("nemotron_18class", "Nemotron 18-Kelas Baru")):
        for sp_name, sp_stats in stats_by_dataset[d_key].items():
            stat_rows.append({
                "dataset_key": d_key,
                "dataset_name": d_label,
                "split": sp_name,
                "sentences": sp_stats["S"],
                "tokens": sp_stats["T"],
                "entities": sp_stats["E"],
                "entity_tokens": sp_stats["T_entity"],
                "avg_tokens_per_sentence": round(sp_stats["avg_tokens"], 2),
                "avg_entities_per_sentence": round(sp_stats["avg_entities"], 2),
                "density_token_pct": round(sp_stats["density_token_pct"], 2),
                "entities_per_100_tokens": round(sp_stats["entities_per_100_tokens"], 2),
                "unique_vocab": sp_stats["unique_vocab"],
                "median_tokens": round(sp_stats["median_tokens"], 1),
                "p95_tokens": round(sp_stats["p95_tokens"], 1),
                "no_entity_sentences": sp_stats["no_entity_sentences"],
                "no_entity_pct": round(sp_stats["no_entity_pct"], 2),
            })
    write_csv_rows(TABLES_DIR / "dataset_statistics.csv", stat_rows)

    class_dist_rows = []
    gold_all_cls = stats_by_dataset["gold"]["all"]["class_counts"]
    gold_train_cls = stats_by_dataset["gold"]["train"]["class_counts"]
    gold_test_cls = stats_by_dataset["gold"]["test"]["class_counts"]
    nem_all_cls = stats_by_dataset["nemotron_18class"]["all"]["class_counts"]
    nem_train_cls = stats_by_dataset["nemotron_18class"]["train"]["class_counts"]
    nem_val_cls = stats_by_dataset["nemotron_18class"]["validation"]["class_counts"]

    for kind in ACTIVE_TYPES:
        class_dist_rows.append({
            "entity_class": kind,
            "gold_all": gold_all_cls[kind],
            "gold_train": gold_train_cls[kind],
            "gold_test": gold_test_cls[kind],
            "nemotron_18class_all": nem_all_cls[kind],
            "nemotron_18class_train": nem_train_cls[kind],
            "nemotron_18class_validation": nem_val_cls[kind],
        })
    write_csv_rows(TABLES_DIR / "per_class_distribution_corpus.csv", class_dist_rows)

    sync_to_drive(TABLES_DIR / "dataset_statistics.csv", "tables/dataset_statistics.csv")
    sync_to_drive(TABLES_DIR / "per_class_distribution_corpus.csv", "tables/per_class_distribution_corpus.csv")
    sync_to_drive(manifest_path, "prepared_manifest.json")

    return prepared, manifest


PREPARED_DATA, PREPARED_MANIFEST = load_and_prepare_corpora()

print("=" * 84)
print("📊 STATISTIK PARTISI KORPUS & PEMERIKSAAN ANTI-KEBOCORAN (ZERO TEST LEAKAGE)")
print(f"   Status Anti-Leakage   : LULUS (Overlap Lintas-Partisi = {PREPARED_MANIFEST['leakage_check']['cross_partition_overlap_groups']})")
print(f"   Eksklusi Kebocoran    : {PREPARED_MANIFEST['nemotron_test_leakage_excluded_count']} kalimat Nemotron terindikasi mirip Test Gold")
print("-" * 84)
print(f"   {'Dataset / Partisi':<30} | {'Kalimat (S)':>11} | {'Token (T)':>11} | {'Entitas (E)':>11} | {'E/Kal':>6} | {'D_tok%':>6}")
print("-" * 84)
for d_key, d_title in (("gold", "NERGRIT Gold"), ("nemotron_18class", "Nemotron 18-Class")):
    for sp in PREPARED_DATA[d_key]:
        st = PREPARED_MANIFEST["statistics"][d_key][sp]
        label_str = f"{d_title} ({sp})"
        print(
            f"   {label_str:<30} | {st['S']:>11,} | {st['T']:>11,} | {st['E']:>11,} | "
            f"{st['avg_entities']:>6.2f} | {st['density_token_pct']:>5.1f}%"
        )
print("-" * 84)
print("🏷️ DISTRIBUSI ENTITAS 18 KELAS (TOTAL KORPUS GOLD vs NEMOTRON 18-KELAS BARU):")
for kind in ACTIVE_TYPES:
    g_cnt = PREPARED_MANIFEST["statistics"]["gold"]["all"]["class_counts"][kind]
    n_cnt = PREPARED_MANIFEST["statistics"]["nemotron_18class"]["all"]["class_counts"][kind]
    g_test = PREPARED_MANIFEST["statistics"]["gold"]["test"]["class_counts"][kind]
    print(f"   - {kind:<4}: Gold All = {g_cnt:>6,} (Test = {g_test:>5,}) | Nemotron 18-Kelas Baru = {n_cnt:>6,}")
print("=" * 84)

In [ ]:
# =============================================================================
# SEL 6: WORD-ALIGNED LOSSLESS CHUNKING (256) & EVALUATOR SPAN CONLL EXACT-MATCH
# =============================================================================
import gc
import inspect

spans = entity_spans


def metrics_from_counts(counts: dict[str, dict[str, int]]) -> dict:
    """Hitung Micro, Macro, Weighted Precision/Recall/F1 dan metrik per kelas dari TP/FP/FN."""
    def score(tp: int, fp: int, fn: int) -> dict:
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
        return {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": tp + fn,
            "predicted": tp + fp,
            "tp": tp,
            "fp": fp,
            "fn": fn,
        }

    per_class = {kind: score(**values) for kind, values in counts.items()}
    micro = score(*(sum(row[key] for row in counts.values()) for key in ("tp", "fp", "fn")))
    total_support = micro["support"]
    result = {"micro_" + key: micro[key] for key in ("precision", "recall", "f1")}
    for key in ("precision", "recall", "f1"):
        result["macro_" + key] = (
            sum(x[key] for x in per_class.values()) / len(per_class) if per_class else 0.0
        )
        result["weighted_" + key] = (
            sum(x[key] * x["support"] for x in per_class.values()) / total_support
            if total_support else 0.0
        )
    result.update({
        "support": total_support,
        "predicted": micro["predicted"],
        "tp": micro["tp"],
        "fp": micro["fp"],
        "fn": micro["fn"],
        "per_class": per_class,
    })
    return result


def span_metrics(
    references: list[list[str]],
    predictions: list[list[str]],
    entity_types: tuple[str, ...] | list[str] = ACTIVE_TYPES,
) -> dict:
    """Evaluasi Exact Whole-Span Match tingkat CoNLL pada himpunan 18 kelas tetap."""
    if len(references) != len(predictions):
        raise ValueError("Jumlah kalimat referensi dan prediksi berbeda!")
    kinds = tuple(entity_types)
    counts = {kind: {"tp": 0, "fp": 0, "fn": 0} for kind in kinds}
    for truth, predicted in zip(references, predictions):
        if len(truth) != len(predicted):
            raise ValueError("Jumlah token kata referensi dan prediksi berbeda!")
        gold_set, pred_set = set(spans(truth)), set(spans(predicted))
        for group, key in ((gold_set & pred_set, "tp"), (pred_set - gold_set, "fp"), (gold_set - pred_set, "fn")):
            for _, _, kind in group:
                if kind not in counts:
                    raise ValueError(f"Tipe entitas tidak dikenal saat evaluasi: {kind}")
                counts[kind][key] += 1
    return metrics_from_counts(counts)


def chunk_record(
    record: dict,
    tokenizer,
    labels: list[str] = LABELS,
    max_length: int = MAX_SEQ_LENGTH,
    avoid_entity_boundaries: bool = True,
) -> list[dict]:
    """Potong kalimat panjang pada batas kata secara lossless (tanpa kehilangan token)."""
    tokens, tags = record["tokens"], record["tags"]
    if not tokens or len(tokens) != len(tags):
        raise ValueError(f"Panjang token/tag tidak valid untuk {record['sentence_id']}")
    unknown = set(tags) - set(labels)
    if unknown:
        raise ValueError(f"Tag model tidak dikenal: {sorted(unknown)}")
    capacity = max_length - tokenizer.num_special_tokens_to_add(pair=False)
    if capacity <= 0:
        raise ValueError("Tidak ada kapasitas token tersisa setelah special tokens")

    full = tokenizer(
        tokens,
        is_split_into_words=True,
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
    )
    word_ids = full.word_ids()
    lengths = [0] * len(tokens)
    for word in word_ids:
        if word is not None:
            lengths[word] += 1

    missing = [index for index, length in enumerate(lengths) if length == 0]
    if missing:
        unknown_word = getattr(tokenizer, "unk_token", None)
        if not unknown_word:
            raise ValueError("Tokenizer tidak memiliki unk_token untuk token kosong/spasi")
        tokens = list(tokens)
        for index in missing:
            tokens[index] = unknown_word
        full = tokenizer(
            tokens,
            is_split_into_words=True,
            add_special_tokens=False,
            truncation=False,
            return_attention_mask=False,
        )
        lengths = [0] * len(tokens)
        for word in full.word_ids():
            if word is not None:
                lengths[word] += 1

    if any(length == 0 or length > capacity for length in lengths):
        raise ValueError(f"Terdapat kata yang melebihi kapasitas subword pada {record['sentence_id']}")

    entity_ranges = spans(tags) if avoid_entity_boundaries else []
    for start, end, _ in entity_ranges:
        if sum(lengths[start:end]) > capacity:
            raise ValueError(f"Rentang entitas melebihi kapasitas input di {record['sentence_id']}:{start}:{end}")

    label2id = {tag: i for i, tag in enumerate(labels)}
    chunks, start = [], 0
    while start < len(tokens):
        end, used = start, 0
        while end < len(tokens) and used + lengths[end] <= capacity:
            used += lengths[end]
            end += 1
        for ent_start, ent_end, _ in entity_ranges:
            if ent_start < end < ent_end:
                end = ent_start
                break
        if end <= start:
            raise ValueError(f"Tidak dapat membentuk chunk lossless untuk {record['sentence_id']}")
        encoded = tokenizer(
            tokens[start:end],
            is_split_into_words=True,
            add_special_tokens=True,
            truncation=False,
        )
        if len(encoded["input_ids"]) > max_length:
            raise ValueError("Panjang subword melampaui max_length saat chunking")
        local_ids = encoded.word_ids()
        aligned, first_positions, previous = [], [], None
        for position, word in enumerate(local_ids):
            if word is None or word == previous:
                aligned.append(-100)
            else:
                aligned.append(label2id[tags[start + word]])
                first_positions.append(position)
            previous = word
        if len(first_positions) != end - start:
            raise ValueError("Tokenizer gagal mempertahankan setiap kata sumber")
        inputs = {
            key: value
            for key, value in encoded.items()
            if key in ("input_ids", "attention_mask", "token_type_ids")
        }
        inputs["labels"] = aligned
        chunks.append({
            "inputs": inputs,
            "sentence_id": record["sentence_id"],
            "start": start,
            "end": end,
            "first_positions": first_positions,
            "unk_source_positions": [idx for idx in missing if start <= idx < end],
        })
        start = end
    return chunks


def recombine_predictions(
    records: list[dict],
    chunks: list[dict],
    predicted_ids: list,
    labels: list[str] = LABELS,
) -> list[list[str]]:
    """Gabungkan kembali prediksi tingkat chunk menjadi prediksi kata utuh tingkat kalimat."""
    if len(chunks) != len(predicted_ids):
        raise ValueError("Jumlah chunk dan prediksi tidak cocok")
    by_id = {row["sentence_id"]: [None] * len(row["tokens"]) for row in records}
    if len(by_id) != len(records):
        raise ValueError("Terdapat duplikasi sentence_id pada records")
    for chunk, ids in zip(chunks, predicted_ids):
        words = by_id[chunk["sentence_id"]]
        for offset, position in enumerate(chunk["first_positions"]):
            absolute = chunk["start"] + offset
            if words[absolute] is not None:
                raise ValueError("Satu kata diprediksi lebih dari satu kali")
            words[absolute] = labels[int(ids[position])]
    if any(tag is None for tags in by_id.values() for tag in tags):
        raise ValueError("Terdapat kata yang belum memiliki hasil prediksi")
    return [by_id[row["sentence_id"]] for row in records]


def _file_hashes(root: Path) -> dict[str, str]:
    root = Path(root)
    result = {}
    for path in sorted(root.rglob("*")):
        if path.is_file():
            rel = str(path.relative_to(root)).replace("\\", "/")
            result[rel] = compute_sha256(path)
    return result


def _trainer_tokenizer_kwarg(tokenizer) -> dict:
    from transformers import Trainer
    sig = inspect.signature(Trainer.__init__).parameters
    return {"processing_class" if "processing_class" in sig else "tokenizer": tokenizer}


def build_training_arguments(config: dict, run_dir: Path, seed: int):
    """Bangun TrainingArguments yang 100% sesuai spesifikasi Tabel 3.2 skripsi."""
    from transformers import TrainingArguments
    sig = inspect.signature(TrainingArguments.__init__).parameters
    use_workers = config["dataloader_num_workers"] if CUDA_AVAILABLE else 0
    values = dict(
        output_dir=str(Path(run_dir) / "checkpoints"),
        per_device_train_batch_size=config["train_batch_size"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],
        per_device_eval_batch_size=config["eval_batch_size"],
        learning_rate=config["learning_rate"],
        weight_decay=config["weight_decay"],
        adam_beta1=config["adam_beta1"],
        adam_beta2=config["adam_beta2"],
        adam_epsilon=config["adam_epsilon"],
        lr_scheduler_type=config["lr_scheduler_type"],
        num_train_epochs=config["num_train_epochs"],
        max_grad_norm=config["max_grad_norm"],
        fp16=bool(config["fp16"] and CUDA_AVAILABLE),
        bf16=bool(config["bf16"] and CUDA_AVAILABLE),
        tf32=False,
        gradient_checkpointing=config["gradient_checkpointing"],
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model=config["metric_for_best_model"],
        greater_is_better=True,
        save_total_limit=1,
        restore_callback_states_from_checkpoint=True,
        seed=seed,
        data_seed=seed,
        report_to="none",
        dataloader_num_workers=use_workers,
        dataloader_pin_memory=bool(CUDA_AVAILABLE),
        dataloader_persistent_workers=bool(use_workers > 0),
        label_smoothing_factor=0.0,
        optim=config["optim"],
        full_determinism=False,
    )
    if "warmup_ratio" in sig:
        values["warmup_ratio"] = config["warmup_ratio"]
    else:
        values["warmup_steps"] = config["warmup_ratio"]
    eval_key = "eval_strategy" if "eval_strategy" in sig else "evaluation_strategy"
    values[eval_key] = "epoch"
    allowed = set(sig)
    return TrainingArguments(**{k: v for k, v in values.items() if k in allowed})


def load_initial_indobert_model(config: dict):
    """Muat model indobenchmark/indobert-large-p1 dengan 37 tag BIO dan dropout=0.1."""
    from transformers import AutoConfig, AutoModelForTokenClassification
    labels = config["labels"]
    model_config = AutoConfig.from_pretrained(
        config["model_checkpoint"],
        revision=config.get("model_revision"),
    )
    model_config.num_labels = len(labels)
    model_config.id2label = dict(enumerate(labels))
    model_config.label2id = {tag: i for i, tag in enumerate(labels)}
    model_config.hidden_dropout_prob = config["dropout"]
    model_config.attention_probs_dropout_prob = config["dropout"]
    model_config.classifier_dropout = config["dropout"]
    init_kwargs = dict(
        pretrained_model_name_or_path=config["model_checkpoint"],
        revision=config.get("model_revision"),
        config=model_config,
        ignore_mismatched_sizes=False,
    )
    sig = inspect.signature(AutoModelForTokenClassification.from_pretrained).parameters
    if "dtype" in sig:
        init_kwargs["dtype"] = torch.float32
    else:
        init_kwargs["torch_dtype"] = torch.float32
    return AutoModelForTokenClassification.from_pretrained(**init_kwargs)


def train_scenario_model(
    dataset_key: str,
    history_prefix: str,
    seed: int,
    train_records: list[dict],
    val_records: list[dict],
    tokenizer,
    config: dict = CONFIG,
    runs_dir: Path = RUNS_DIR,
) -> dict:
    """Latih atau pulihkan model untuk satu skenario dan cadangkan otomatis ke Google Drive."""
    from datasets import Dataset
    from transformers import DataCollatorForTokenClassification, EarlyStoppingCallback, Trainer

    run_id = f"{dataset_key}_s{seed}"
    run_dir = runs_dir / run_id
    drive_run_dir = DRIVE_BASE_DIR / "runs" / run_id
    manifest_path = run_dir / "run_manifest.json"
    best_model_dir = run_dir / "best_model"

    train_hash = stable_json_hash([
        {"sentence_id": r["sentence_id"], "tokens": r["tokens"], "tags": r["tags"]}
        for r in train_records
    ])
    val_hash = stable_json_hash([
        {"sentence_id": r["sentence_id"], "tokens": r["tokens"], "tags": r["tags"]}
        for r in val_records
    ])

    # 1. Cek apakah run sudah selesai di lokal atau di Google Drive (mendukung auto-resume penuh)
    if not manifest_path.is_file() and (drive_run_dir / "run_manifest.json").is_file():
        try:
            drive_manifest = json.loads((drive_run_dir / "run_manifest.json").read_text(encoding="utf-8"))
            if (
                drive_manifest.get("status") == "complete"
                and drive_manifest.get("train_hash") == train_hash
                and (drive_run_dir / "best_model" / "config.json").is_file()
            ):
                print(f"☁️ Memulihkan model {run_id} yang telah selesai dari Google Drive...")
                shutil.copytree(drive_run_dir, run_dir, dirs_exist_ok=True)
        except Exception as exc:
            print(f"⚠️ Gagal memulihkan {run_id} dari Google Drive: {exc}")

    if manifest_path.is_file():
        existing = json.loads(manifest_path.read_text(encoding="utf-8"))
        if (
            existing.get("status") == "complete"
            and existing.get("train_hash") == train_hash
            and (best_model_dir / "config.json").is_file()
        ):
            hist_csv_path = HISTORIES_DIR / f"{history_prefix}_s{seed}.csv"
            hist_json_path = HISTORIES_DIR / f"{history_prefix}_s{seed}.json"
            if (run_dir / "training_history.csv").is_file() and not hist_csv_path.is_file():
                shutil.copy2(run_dir / "training_history.csv", hist_csv_path)
            if (run_dir / "training_history.json").is_file() and not hist_json_path.is_file():
                shutil.copy2(run_dir / "training_history.json", hist_json_path)
            print(f"✅ Run {run_id} sudah selesai dan terverifikasi. Melewati pelatihan ulang.")
            return existing

    # 2. Chunking kalimat latih & validasi
    print(f"✂️ Melakukan word-aligned chunking (max_length={config['max_seq_length']}) untuk {run_id}...")
    train_chunks = []
    for rec in train_records:
        train_chunks.extend(
            chunk_record(rec, tokenizer, config["labels"], config["max_seq_length"], avoid_entity_boundaries=True)
        )
    val_chunks = []
    for rec in val_records:
        val_chunks.extend(
            chunk_record(rec, tokenizer, config["labels"], config["max_seq_length"], avoid_entity_boundaries=False)
        )
    print(
        f"   Train: {len(train_records):,} kalimat -> {len(train_chunks):,} chunks | "
        f"Validasi: {len(val_records):,} kalimat -> {len(val_chunks):,} chunks"
    )

    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_root = run_dir / "checkpoints"
    resume_ckpt = None
    if ckpt_root.is_dir():
        ckpts = sorted(
            (p for p in ckpt_root.glob("checkpoint-*") if p.is_dir() and re.fullmatch(r"checkpoint-\d+", p.name)),
            key=lambda p: int(p.name.rsplit("-", 1)[-1]),
        )
        if ckpts:
            resume_ckpt = str(ckpts[-1])
            print(f"🔄 Melanjutkan pelatihan {run_id} dari checkpoint: {resume_ckpt}")

    configure_reproducibility(seed)
    model = load_initial_indobert_model(config)

    labels = config["labels"]
    kinds = tuple(config["active_types"])

    def compute_metrics(eval_pred):
        logits = eval_pred.predictions[0] if isinstance(eval_pred.predictions, tuple) else eval_pred.predictions
        pred_ids = logits.argmax(axis=-1)
        pred_tags = recombine_predictions(val_records, val_chunks, pred_ids, labels)
        scores = span_metrics([row["tags"] for row in val_records], pred_tags, kinds)
        return {k: v for k, v in scores.items() if k != "per_class"}

    trainer = Trainer(
        model=model,
        args=build_training_arguments(config, run_dir, seed),
        train_dataset=Dataset.from_list([c["inputs"] for c in train_chunks]),
        eval_dataset=Dataset.from_list([c["inputs"] for c in val_chunks]),
        data_collator=DataCollatorForTokenClassification(
            tokenizer, pad_to_multiple_of=config["pad_to_multiple_of"]
        ),
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=config["early_stopping_patience"],
                early_stopping_threshold=config["early_stopping_threshold"],
            )
        ],
        **_trainer_tokenizer_kwarg(tokenizer),
    )

    print(f"🚀 Memulai fine-tuning {run_id} pada {GPU_NAME} ({PRECISION_MODE.upper()})...")
    t_start = time.time()
    train_output = trainer.train(resume_from_checkpoint=resume_ckpt)
    elapsed_sec = time.time() - t_start

    # 3. Simpan model terbaik, tokenizer, dan riwayat pelatihan
    best_model_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(best_model_dir))
    tokenizer.save_pretrained(str(best_model_dir))

    log_history = list(trainer.state.log_history)
    (run_dir / "training_history.json").write_text(
        json.dumps(log_history, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    write_csv_rows(run_dir / "training_history.csv", log_history)

    # Salin ke direktori histories/ untuk arsip akhir
    hist_csv_path = HISTORIES_DIR / f"{history_prefix}_s{seed}.csv"
    hist_json_path = HISTORIES_DIR / f"{history_prefix}_s{seed}.json"
    write_csv_rows(hist_csv_path, log_history)
    hist_json_path.write_text(json.dumps(log_history, indent=2, ensure_ascii=False), encoding="utf-8")

    trainer_state_dict = (
        trainer.state.to_dict() if hasattr(trainer.state, "to_dict") else vars(trainer.state)
    )
    (run_dir / "trainer_state.json").write_text(
        json.dumps(trainer_state_dict, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    manifest = {
        "status": "complete",
        "run_id": run_id,
        "dataset": dataset_key,
        "history_prefix": history_prefix,
        "seed": seed,
        "model_checkpoint": config["model_checkpoint"],
        "precision_mode": PRECISION_MODE,
        "gpu_name": GPU_NAME,
        "train_sentences": len(train_records),
        "train_chunks": len(train_chunks),
        "train_tokens": sum(len(r["tokens"]) for r in train_records),
        "train_entities": sum(len(spans(r["tags"])) for r in train_records),
        "validation_sentences": len(val_records),
        "validation_chunks": len(val_chunks),
        "train_hash": train_hash,
        "validation_hash": val_hash,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_validation_macro_f1": float(trainer.state.best_metric or 0.0),
        "epochs_completed": float(trainer.state.epoch or 0.0),
        "optimizer_steps": int(trainer.state.global_step or 0),
        "elapsed_seconds": round(elapsed_sec, 2),
        "elapsed_minutes": round(elapsed_sec / 60.0, 2),
        "training_metrics": train_output.metrics,
        "model_files": _file_hashes(best_model_dir),
    }
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

    # 4. Bersihkan checkpoint optimizer sementara setelah best_model aman
    if ckpt_root.is_dir():
        shutil.rmtree(ckpt_root, ignore_errors=True)

    del trainer, model
    gc.collect()
    if CUDA_AVAILABLE:
        torch.cuda.empty_cache()

    # 5. Cadangkan otomatis ke Google Drive
    print(f"☁️ Mencadangkan artefak {run_id} ke Google Drive...")
    for fname in ("run_manifest.json", "training_history.json", "training_history.csv", "trainer_state.json"):
        sync_to_drive(run_dir / fname, f"runs/{run_id}/{fname}")
    sync_to_drive(hist_csv_path, f"histories/{hist_csv_path.name}")
    sync_to_drive(hist_json_path, f"histories/{hist_json_path.name}")
    if config.get("save_best_weights_to_drive", True):
        sync_to_drive(best_model_dir, f"runs/{run_id}/best_model", verify_hash=False)

    return manifest


def predict_with_model(
    model_dir: Path,
    records: list[dict],
    tokenizer,
    config: dict = CONFIG,
    work_dir: Path | None = None,
) -> list[list[str]]:
    """Jalankan inferensi label-blind pada himpunan uji dan gabungkan kembali ke token kata."""
    from datasets import Dataset
    from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification, Trainer, TrainingArguments

    blind_records = [{**rec, "tags": ["O"] * len(rec["tokens"])} for rec in records]
    chunks = []
    for rec in blind_records:
        chunks.extend(
            chunk_record(rec, tokenizer, config["labels"], config["max_seq_length"], avoid_entity_boundaries=False)
        )

    init_kwargs = dict(pretrained_model_name_or_path=str(model_dir))
    sig = inspect.signature(AutoModelForTokenClassification.from_pretrained).parameters
    if "dtype" in sig:
        init_kwargs["dtype"] = torch.float32
    else:
        init_kwargs["torch_dtype"] = torch.float32
    model = AutoModelForTokenClassification.from_pretrained(**init_kwargs)

    tmp_eval_dir = str(work_dir or (OUTPUT_DIR / "eval_runtime"))
    eval_args = TrainingArguments(
        output_dir=tmp_eval_dir,
        per_device_eval_batch_size=config["eval_batch_size"],
        report_to="none",
        fp16=bool(config["fp16"] and CUDA_AVAILABLE),
        bf16=bool(config["bf16"] and CUDA_AVAILABLE),
        tf32=False,
        dataloader_num_workers=0,
    )
    trainer = Trainer(
        model=model,
        args=eval_args,
        data_collator=DataCollatorForTokenClassification(
            tokenizer, pad_to_multiple_of=config["pad_to_multiple_of"]
        ),
        **_trainer_tokenizer_kwarg(tokenizer),
    )
    output = trainer.predict(Dataset.from_list([c["inputs"] for c in chunks]))
    logits = output.predictions[0] if isinstance(output.predictions, tuple) else output.predictions
    pred_ids = logits.argmax(axis=-1)
    predictions = recombine_predictions(records, chunks, pred_ids, config["labels"])

    del trainer, model, output, logits
    gc.collect()
    if CUDA_AVAILABLE:
        torch.cuda.empty_cache()
    return predictions


def export_predictions(base_path: Path, records: list[dict], predictions: list[list[str]]) -> tuple[Path, Path]:
    """Ekspor hasil prediksi token demi token ke format .jsonl dan .conll."""
    base_path = Path(base_path)
    base_path.parent.mkdir(parents=True, exist_ok=True)
    jsonl_path = base_path.with_suffix(".jsonl")
    conll_path = base_path.with_suffix(".conll")
    with jsonl_path.open("w", encoding="utf-8") as jf, conll_path.open("w", encoding="utf-8") as cf:
        for rec, pred_tags in zip(records, predictions):
            jf.write(
                json.dumps(
                    {
                        "sentence_id": rec["sentence_id"],
                        "tokens": rec["tokens"],
                        "reference_tags": rec["tags"],
                        "predicted_tags": pred_tags,
                        "reference_entities": [
                            {"start": s, "end": e, "type": k, "text": " ".join(rec["tokens"][s:e])}
                            for s, e, k in spans(rec["tags"])
                        ],
                        "predicted_entities": [
                            {"start": s, "end": e, "type": k, "text": " ".join(rec["tokens"][s:e])}
                            for s, e, k in spans(pred_tags)
                        ],
                    },
                    ensure_ascii=False,
                )
                + "\n"
            )
            cf.write(f"# sentence_id = {rec['sentence_id']}\n")
            for tok, ref_tag, pred_tag in zip(rec["tokens"], rec["tags"], pred_tags):
                cf.write(f"{tok}\t{ref_tag}\t{pred_tag}\n")
            cf.write("\n")
    return jsonl_path, conll_path


# Uji Mandiri (Self-Test) Evaluator Span CoNLL
_dummy_ref = [["B-PER", "I-PER", "O", "B-GPE", "O", "B-NOR"]]
_dummy_pred = [["B-PER", "I-PER", "O", "B-LOC", "O", "B-NOR"]]
_dummy_score = span_metrics(_dummy_ref, _dummy_pred, ACTIVE_TYPES)
assert _dummy_score["tp"] == 2 and _dummy_score["fp"] == 1 and _dummy_score["fn"] == 1
print("✅ Fungsi chunk_record, recombine_predictions, dan span_metrics siap digunakan.")

In [ ]:
# =============================================================================
# SEL 7: PELATIHAN MODEL SKENARIO 1 — NERGRIT GOLD BASELINE (SEED 42)
# =============================================================================
from transformers import AutoTokenizer

# Kunci revisi model & muat Fast Tokenizer IndoBERT-Large-p1
if CONFIG.get("model_revision") is None:
    try:
        from huggingface_hub import HfApi
        CONFIG["model_revision"] = HfApi().model_info(MODEL_CHECKPOINT).sha
        print(f"🔒 Revisi model terkunci pada commit SHA: {CONFIG['model_revision']}")
    except Exception as exc:
        print(f"ℹ️ Menggunakan revisi default cabang utama untuk {MODEL_CHECKPOINT} ({exc})")

TOKENIZER = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    revision=CONFIG.get("model_revision"),
    use_fast=True,
)
if not TOKENIZER.is_fast:
    raise RuntimeError("Word-aligned token classification membutuhkan Fast Tokenizer (Rust).")

tokenizer_lock_dir = OUTPUT_DIR / "tokenizer_lock"
tokenizer_lock_dir.mkdir(parents=True, exist_ok=True)
TOKENIZER.save_pretrained(str(tokenizer_lock_dir))


def print_training_history_summary(run_manifest: dict, run_dir: Path) -> None:
    """Tampilkan ringkasan per-epoch dari riwayat pelatihan model."""
    hist_path = run_dir / "training_history.json"
    if not hist_path.is_file():
        return
    history = json.loads(hist_path.read_text(encoding="utf-8"))
    eval_entries = [row for row in history if "eval_macro_f1" in row or "eval_loss" in row]
    train_losses = {
        round(float(row["epoch"]), 2): row["loss"]
        for row in history
        if "loss" in row and "eval_loss" not in row and "epoch" in row
    }
    print("-" * 78)
    print(f"   {'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Micro-F1':>12} | {'Val Macro-F1':>12}")
    print("-" * 78)
    for row in eval_entries:
        ep = round(float(row.get("epoch", 0.0)), 2)
        t_loss = train_losses.get(ep)
        t_loss_str = f"{t_loss:.4f}" if isinstance(t_loss, (int, float)) else "—"
        v_loss = row.get("eval_loss", 0.0)
        v_mic = row.get("eval_micro_f1", 0.0) * 100.0
        v_mac = row.get("eval_macro_f1", 0.0) * 100.0
        print(f"   {ep:>5.1f} | {t_loss_str:>10} | {v_loss:>10.4f} | {v_mic:>11.2f}% | {v_mac:>11.2f}%")
    print("-" * 78)


GOLD_RUN_MANIFESTS = {}
for seed in TRAINING_SEEDS:
    print("\n" + "=" * 78)
    print(f"🏅 SKENARIO 1: PELATIHAN NERGRIT GOLD BASELINE (SEED = {seed})")
    print("=" * 78)
    manifest = train_scenario_model(
        dataset_key="gold",
        history_prefix="gold",
        seed=seed,
        train_records=PREPARED_DATA["gold"]["train"],
        val_records=PREPARED_DATA["gold"]["validation"],
        tokenizer=TOKENIZER,
        config=CONFIG,
        runs_dir=RUNS_DIR,
    )
    GOLD_RUN_MANIFESTS[seed] = manifest
    print_training_history_summary(manifest, RUNS_DIR / manifest["run_id"])
    print(
        f"✅ Selesai {manifest['run_id']}: Best Val Macro-F1 = {manifest['best_validation_macro_f1'] * 100:.2f}% | "
        f"Epochs = {manifest['epochs_completed']} | Steps = {manifest['optimizer_steps']} | "
        f"Durasi = {manifest['elapsed_minutes']:.2f} menit"
    )

In [ ]:
# =============================================================================
# SEL 8: PELATIHAN MODEL SKENARIO 2 — NEMOTRON 18-KELAS BARU (SEED 42)
# =============================================================================
NEMOTRON_RUN_MANIFESTS = {}

for seed in TRAINING_SEEDS:
    print("\n" + "=" * 78)
    print(f"🤖 SKENARIO 2: PELATIHAN KORPUS SINTETIS NEMOTRON 18-KELAS BARU (SEED = {seed})")
    print("=" * 78)
    manifest = train_scenario_model(
        dataset_key="nemotron_18class",
        history_prefix="nemotron",
        seed=seed,
        train_records=PREPARED_DATA["nemotron_18class"]["train"],
        val_records=PREPARED_DATA["nemotron_18class"]["validation"],
        tokenizer=TOKENIZER,
        config=CONFIG,
        runs_dir=RUNS_DIR,
    )
    NEMOTRON_RUN_MANIFESTS[seed] = manifest

    # Buat salinan alias riwayat dengan nama eksplisit nemotron_18class_s{seed}
    src_csv = HISTORIES_DIR / f"nemotron_s{seed}.csv"
    src_json = HISTORIES_DIR / f"nemotron_s{seed}.json"
    alias_csv = HISTORIES_DIR / f"nemotron_18class_s{seed}.csv"
    alias_json = HISTORIES_DIR / f"nemotron_18class_s{seed}.json"
    if src_csv.is_file():
        shutil.copy2(src_csv, alias_csv)
        sync_to_drive(alias_csv, f"histories/{alias_csv.name}")
    if src_json.is_file():
        shutil.copy2(src_json, alias_json)
        sync_to_drive(alias_json, f"histories/{alias_json.name}")

    print_training_history_summary(manifest, RUNS_DIR / manifest["run_id"])
    print(
        f"✅ Selesai {manifest['run_id']}: Best Val Macro-F1 = {manifest['best_validation_macro_f1'] * 100:.2f}% | "
        f"Epochs = {manifest['epochs_completed']} | Steps = {manifest['optimizer_steps']} | "
        f"Durasi = {manifest['elapsed_minutes']:.2f} menit"
    )

In [ ]:
# =============================================================================
# SEL 9: EVALUASI HEAD-TO-HEAD PADA NERGRIT GOLD TEST SET (AGREGAT & PER KELAS)
# =============================================================================

# Referensi Skor Bab 4 Sebelum Verifikasi Ulang 18-Kelas (untuk Perbandingan Dampak Pembersihan)
HISTORICAL_BAB4_REFERENCE = {
    "gold_bab4_seed42": {
        "micro_precision": 0.8365,
        "micro_recall": 0.8422,
        "micro_f1": 0.8393,
        "macro_f1": 0.7552,
        "weighted_f1": 0.8380,
        "support": 8890,
    },
    "nemotron_old_seed42": {
        "micro_precision": 0.5474,
        "micro_recall": 0.5111,
        "micro_f1": 0.5286,
        "macro_f1": 0.4181,
        "weighted_f1": 0.5083,
        "support": 8890,
    },
    "nemotron_old_mean_3seeds": {
        "micro_precision": 0.5413,
        "micro_recall": 0.5056,
        "micro_f1": 0.5228,
        "macro_f1": 0.4176,
        "weighted_f1": 0.5030,
        "support": 8890,
        "per_class_f1": {
            "CRD": 0.6617, "DAT": 0.4313, "EVT": 0.5668, "FAC": 0.2067,
            "GPE": 0.7485, "LAN": 0.0000, "LAW": 0.3671, "LOC": 0.1550,
            "MON": 0.7298, "NOR": 0.0121, "ORD": 0.5691, "ORG": 0.4423,
            "PER": 0.7327, "PRC": 0.8911, "PRD": 0.2673, "QTY": 0.1955,
            "TIM": 0.4155, "WOA": 0.1237,
        },
    },
    "gold_bab4_mean_3seeds_per_class_f1": {
        "CRD": 0.8880, "DAT": 0.9471, "EVT": 0.7059, "FAC": 0.4585,
        "GPE": 0.8912, "LAN": 0.0000, "LAW": 0.7110, "LOC": 0.6227,
        "MON": 0.9712, "NOR": 0.7983, "ORD": 0.7619, "ORG": 0.7520,
        "PER": 0.9657, "PRC": 0.9441, "PRD": 0.6875, "QTY": 0.6451,
        "TIM": 0.9430, "WOA": 1.0000,
    },
}

ENTITY_CLASS_DESCRIPTIONS = {
    "CRD": "Bilangan Kardinal (Cardinal)",
    "DAT": "Tanggal Kalender (Date)",
    "EVT": "Peristiwa / Acara (Event)",
    "FAC": "Fasilitas Bangunan / Infrastruktur (Facility)",
    "GPE": "Entitas Geo-Politik (Negara, Kota, Provinsi)",
    "LAN": "Bahasa Tertutur / Tertulis (Language)",
    "LAW": "Hukum dan Perundang-undangan (Law)",
    "LOC": "Lokasi Non-Geopolitik (Gunung, Sungai, Wilayah)",
    "MON": "Jumlah Nilai Uang (Money)",
    "NOR": "Kebangsaan, Agama, Suku, Jabatan Politik/Negara (NORP)",
    "ORD": "Bilangan Ordinal / Tingkatan (Ordinal)",
    "ORG": "Organisasi, Perusahaan, Lembaga (Organization)",
    "PER": "Nama Individu / Tokoh Manusia (Person)",
    "PRC": "Nilai Persentase (Percent)",
    "PRD": "Nama Produk / Sarana Transportasi / Komoditas (Product)",
    "QTY": "Besaran Ukuran Fisik / Durasi Kuantitas (Quantity)",
    "TIM": "Waktu Jam Spesifik (Time)",
    "WOA": "Karya Cipta Seni / Sastra / Film (Work of Art)",
}


def evaluate_all_scenarios_on_gold_test(
    test_records: list[dict],
    tokenizer,
    seeds: list[int] = TRAINING_SEEDS,
    runs_dir: Path = RUNS_DIR,
    predictions_dir: Path = PREDICTIONS_DIR,
    tables_dir: Path = TABLES_DIR,
) -> dict:
    """Jalankan inferensi kedua model pada Gold Test Set dan susun tabel komparasi lengkap."""
    ref_tags = [rec["tags"] for rec in test_records]
    evaluation_results = {"gold": {}, "nemotron_18class": {}}
    predictions_by_run = {"gold": {}, "nemotron_18class": {}}

    scenario_specs = (
        ("gold", "gold", "NERGRIT Gold Baseline"),
        ("nemotron_18class", "nemotron", "Nemotron 18-Kelas Baru (Terverifikasi)"),
    )

    for dataset_key, file_prefix, label_title in scenario_specs:
        for seed in seeds:
            run_id = f"{dataset_key}_s{seed}"
            model_dir = runs_dir / run_id / "best_model"
            if not (model_dir / "config.json").is_file():
                drive_model_dir = DRIVE_BASE_DIR / "runs" / run_id / "best_model"
                if (drive_model_dir / "config.json").is_file():
                    print(f"☁️ Memulihkan best_model {run_id} dari Google Drive...")
                    shutil.copytree(drive_model_dir, model_dir, dirs_exist_ok=True)
                else:
                    raise FileNotFoundError(f"Model terbaik tidak ditemukan untuk {run_id} di {model_dir}")

            print(f"🔍 Mengevaluasi {label_title} ({run_id}) pada NERGRIT Gold Test Set ({len(test_records):,} kalimat)...")
            pred_tags = predict_with_model(
                model_dir=model_dir,
                records=test_records,
                tokenizer=tokenizer,
                config=CONFIG,
                work_dir=OUTPUT_DIR / "eval_runtime" / run_id,
            )
            predictions_by_run[dataset_key][seed] = pred_tags

            base_pred_path = predictions_dir / f"{file_prefix}_s{seed}_test_predictions"
            jsonl_p, conll_p = export_predictions(base_pred_path, test_records, pred_tags)
            sync_to_drive(jsonl_p, f"predictions/{jsonl_p.name}")
            sync_to_drive(conll_p, f"predictions/{conll_p.name}")

            if dataset_key == "nemotron_18class":
                alias_base = predictions_dir / f"nemotron_18class_s{seed}_test_predictions"
                a_jsonl, a_conll = export_predictions(alias_base, test_records, pred_tags)
                sync_to_drive(a_jsonl, f"predictions/{a_jsonl.name}")
                sync_to_drive(a_conll, f"predictions/{a_conll.name}")

            metrics = span_metrics(ref_tags, pred_tags, ACTIVE_TYPES)
            evaluation_results[dataset_key][seed] = metrics

    # Susun Tabel Komparasi Agregat (overall_comparison.csv)
    overall_rows = []
    metric_keys = (
        "micro_precision", "micro_recall", "micro_f1",
        "macro_precision", "macro_recall", "macro_f1",
        "weighted_precision", "weighted_recall", "weighted_f1",
    )

    for dataset_key, file_prefix, label_title in scenario_specs:
        for seed in seeds:
            m = evaluation_results[dataset_key][seed]
            row = {
                "scenario_key": dataset_key,
                "scenario_name": label_title,
                "seed": str(seed),
                "test_sentences": len(test_records),
                "support": m["support"],
                "predicted": m["predicted"],
                "tp": m["tp"],
                "fp": m["fp"],
                "fn": m["fn"],
            }
            for mk in metric_keys:
                row[mk] = round(m[mk], 6)
                row[mk + "_pct"] = round(m[mk] * 100.0, 2)
            overall_rows.append(row)

        if len(seeds) >= 1:
            mean_row = {
                "scenario_key": dataset_key,
                "scenario_name": f"{label_title} (Mean)",
                "seed": "mean",
                "test_sentences": len(test_records),
                "support": evaluation_results[dataset_key][seeds[0]]["support"],
                "predicted": round(statistics.mean(evaluation_results[dataset_key][s]["predicted"] for s in seeds), 1),
                "tp": round(statistics.mean(evaluation_results[dataset_key][s]["tp"] for s in seeds), 1),
                "fp": round(statistics.mean(evaluation_results[dataset_key][s]["fp"] for s in seeds), 1),
                "fn": round(statistics.mean(evaluation_results[dataset_key][s]["fn"] for s in seeds), 1),
            }
            for mk in metric_keys:
                vals = [evaluation_results[dataset_key][s][mk] for s in seeds]
                mean_val = statistics.mean(vals)
                std_val = statistics.stdev(vals) if len(vals) > 1 else 0.0
                mean_row[mk] = round(mean_val, 6)
                mean_row[mk + "_pct"] = round(mean_val * 100.0, 2)
                mean_row[mk + "_std_pct"] = round(std_val * 100.0, 2)
            overall_rows.append(mean_row)

    # Tambahkan baris referensi Nemotron Lama (Bab 4 Sebelum Verifikasi 18-Kelas) untuk perbandingan langsung
    old_s42 = HISTORICAL_BAB4_REFERENCE["nemotron_old_seed42"]
    overall_rows.append({
        "scenario_key": "nemotron_old_bab4_ref",
        "scenario_name": "Nemotron Lama (Referensi Bab 4 Sebelum Verifikasi 18-Kelas)",
        "seed": "42",
        "test_sentences": len(test_records),
        "support": old_s42["support"],
        "predicted": 0,
        "tp": 0,
        "fp": 0,
        "fn": 0,
        "micro_precision": old_s42["micro_precision"],
        "micro_recall": old_s42["micro_recall"],
        "micro_f1": old_s42["micro_f1"],
        "macro_precision": 0.0,
        "macro_recall": 0.0,
        "macro_f1": old_s42["macro_f1"],
        "weighted_precision": 0.0,
        "weighted_recall": 0.0,
        "weighted_f1": old_s42["weighted_f1"],
        "micro_precision_pct": round(old_s42["micro_precision"] * 100.0, 2),
        "micro_recall_pct": round(old_s42["micro_recall"] * 100.0, 2),
        "micro_f1_pct": round(old_s42["micro_f1"] * 100.0, 2),
        "macro_precision_pct": 0.0,
        "macro_recall_pct": 0.0,
        "macro_f1_pct": round(old_s42["macro_f1"] * 100.0, 2),
        "weighted_precision_pct": 0.0,
        "weighted_recall_pct": 0.0,
        "weighted_f1_pct": round(old_s42["weighted_f1"] * 100.0, 2),
    })

    # Hitung Delta (Nemotron 18-Kelas Baru vs Nemotron Lama & vs Gold Baseline)
    primary_seed = seeds[0]
    gold_m = evaluation_results["gold"][primary_seed]
    nem_m = evaluation_results["nemotron_18class"][primary_seed]

    delta_vs_old = {
        "micro_precision_pct_delta": round((nem_m["micro_precision"] - old_s42["micro_precision"]) * 100.0, 2),
        "micro_recall_pct_delta": round((nem_m["micro_recall"] - old_s42["micro_recall"]) * 100.0, 2),
        "micro_f1_pct_delta": round((nem_m["micro_f1"] - old_s42["micro_f1"]) * 100.0, 2),
        "macro_f1_pct_delta": round((nem_m["macro_f1"] - old_s42["macro_f1"]) * 100.0, 2),
        "weighted_f1_pct_delta": round((nem_m["weighted_f1"] - old_s42["weighted_f1"]) * 100.0, 2),
    }
    delta_vs_gold = {
        "micro_precision_pct_delta": round((nem_m["micro_precision"] - gold_m["micro_precision"]) * 100.0, 2),
        "micro_recall_pct_delta": round((nem_m["micro_recall"] - gold_m["micro_recall"]) * 100.0, 2),
        "micro_f1_pct_delta": round((nem_m["micro_f1"] - gold_m["micro_f1"]) * 100.0, 2),
        "macro_f1_pct_delta": round((nem_m["macro_f1"] - gold_m["macro_f1"]) * 100.0, 2),
        "weighted_f1_pct_delta": round((nem_m["weighted_f1"] - gold_m["weighted_f1"]) * 100.0, 2),
    }

    # Susun Tabel Per Kelas 18 Kelas (per_class_18_comparison.csv)
    per_class_rows = []
    old_per_class = HISTORICAL_BAB4_REFERENCE["nemotron_old_mean_3seeds"]["per_class_f1"]
    for kind in ACTIVE_TYPES:
        g_cls = gold_m["per_class"][kind]
        n_cls = nem_m["per_class"][kind]
        old_f1 = old_per_class.get(kind, 0.0)
        per_class_rows.append({
            "entity_class": kind,
            "description": ENTITY_CLASS_DESCRIPTIONS.get(kind, kind),
            "support": g_cls["support"],
            "gold_precision_pct": round(g_cls["precision"] * 100.0, 2),
            "gold_recall_pct": round(g_cls["recall"] * 100.0, 2),
            "gold_f1_pct": round(g_cls["f1"] * 100.0, 2),
            "gold_tp": g_cls["tp"],
            "gold_fp": g_cls["fp"],
            "gold_fn": g_cls["fn"],
            "nemotron_18class_precision_pct": round(n_cls["precision"] * 100.0, 2),
            "nemotron_18class_recall_pct": round(n_cls["recall"] * 100.0, 2),
            "nemotron_18class_f1_pct": round(n_cls["f1"] * 100.0, 2),
            "nemotron_18class_tp": n_cls["tp"],
            "nemotron_18class_fp": n_cls["fp"],
            "nemotron_18class_fn": n_cls["fn"],
            "nemotron_old_bab4_f1_pct": round(old_f1 * 100.0, 2),
            "delta_new_vs_old_nemotron_f1_pct": round((n_cls["f1"] - old_f1) * 100.0, 2),
            "delta_new_nemotron_vs_gold_f1_pct": round((n_cls["f1"] - g_cls["f1"]) * 100.0, 2),
        })

    overall_csv = tables_dir / "overall_comparison.csv"
    overall_json = tables_dir / "overall_comparison.json"
    per_class_csv = tables_dir / "per_class_18_comparison.csv"
    per_class_json = tables_dir / "per_class_18_comparison.json"

    write_csv_rows(overall_csv, overall_rows)
    overall_json.write_text(
        json.dumps(
            {
                "rows": overall_rows,
                "delta_nemotron_18class_vs_old_nemotron": delta_vs_old,
                "delta_nemotron_18class_vs_gold": delta_vs_gold,
            },
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )
    write_csv_rows(per_class_csv, per_class_rows)
    per_class_json.write_text(json.dumps(per_class_rows, indent=2, ensure_ascii=False), encoding="utf-8")

    for path_item in (overall_csv, overall_json, per_class_csv, per_class_json):
        sync_to_drive(path_item, f"tables/{path_item.name}")

    return {
        "evaluation_results": evaluation_results,
        "predictions_by_run": predictions_by_run,
        "overall_rows": overall_rows,
        "per_class_rows": per_class_rows,
        "delta_vs_old": delta_vs_old,
        "delta_vs_gold": delta_vs_gold,
    }


EVAL_BUNDLE = evaluate_all_scenarios_on_gold_test(
    test_records=PREPARED_DATA["gold"]["test"],
    tokenizer=TOKENIZER,
    seeds=TRAINING_SEEDS,
)

_s0 = TRAINING_SEEDS[0]
_g_res = EVAL_BUNDLE["evaluation_results"]["gold"][_s0]
_n_res = EVAL_BUNDLE["evaluation_results"]["nemotron_18class"][_s0]
_old_res = HISTORICAL_BAB4_REFERENCE["nemotron_old_seed42"]

print("\n" + "=" * 92)
print("🏆 HASIL EVALUASI UTAMA PADA NERGRIT GOLD TEST SET (EXACT SPAN MATCH - SEED 42)")
print("=" * 92)
print(
    f"   {'Skenario Model':<42} | {'Presisi':>8} | {'Recall':>8} | {'Micro-F1':>9} | {'Macro-F1':>9} | {'Weighted-F1':>11}"
)
print("-" * 92)
print(
    f"   {'1. NERGRIT Gold Baseline (Seed 42)':<42} | "
    f"{_g_res['micro_precision']*100:>7.2f}% | {_g_res['micro_recall']*100:>7.2f}% | "
    f"{_g_res['micro_f1']*100:>8.2f}% | {_g_res['macro_f1']*100:>8.2f}% | {_g_res['weighted_f1']*100:>10.2f}%"
)
print(
    f"   {'2. Nemotron 18-Kelas Baru (Seed 42)':<42} | "
    f"{_n_res['micro_precision']*100:>7.2f}% | {_n_res['micro_recall']*100:>7.2f}% | "
    f"{_n_res['micro_f1']*100:>8.2f}% | {_n_res['macro_f1']*100:>8.2f}% | {_n_res['weighted_f1']*100:>10.2f}%"
)
print(
    f"   {'3. Nemotron Lama (Bab 4 Seed 42 Ref)':<42} | "
    f"{_old_res['micro_precision']*100:>7.2f}% | {_old_res['micro_recall']*100:>7.2f}% | "
    f"{_old_res['micro_f1']*100:>8.2f}% | {_old_res['macro_f1']*100:>8.2f}% | {_old_res['weighted_f1']*100:>10.2f}%"
)
print("-" * 92)
print(
    f"   {'Δ (Nemotron 18-Kelas Baru vs Nemotron Lama)':<42} | "
    f"{EVAL_BUNDLE['delta_vs_old']['micro_precision_pct_delta']:>+7.2f}% | "
    f"{EVAL_BUNDLE['delta_vs_old']['micro_recall_pct_delta']:>+7.2f}% | "
    f"{EVAL_BUNDLE['delta_vs_old']['micro_f1_pct_delta']:>+8.2f}% | "
    f"{EVAL_BUNDLE['delta_vs_old']['macro_f1_pct_delta']:>+8.2f}% | "
    f"{EVAL_BUNDLE['delta_vs_old']['weighted_f1_pct_delta']:>+10.2f}%"
)
print(
    f"   {'Δ (Nemotron 18-Kelas Baru vs Gold Baseline)':<42} | "
    f"{EVAL_BUNDLE['delta_vs_gold']['micro_precision_pct_delta']:>+7.2f}% | "
    f"{EVAL_BUNDLE['delta_vs_gold']['micro_recall_pct_delta']:>+7.2f}% | "
    f"{EVAL_BUNDLE['delta_vs_gold']['micro_f1_pct_delta']:>+8.2f}% | "
    f"{EVAL_BUNDLE['delta_vs_gold']['macro_f1_pct_delta']:>+8.2f}% | "
    f"{EVAL_BUNDLE['delta_vs_gold']['weighted_f1_pct_delta']:>+10.2f}%"
)
print("=" * 92)

print("\n📋 RINCIAN SKOR PER KELAS (18 KELAS ENTITAS PADA GOLD TEST SET):")
print("-" * 92)
print(
    f"   {'Kelas':<5} | {'Support':>7} | {'Gold F1':>8} | {'Nemotron Baru F1':>16} | "
    f"{'Nemotron Lama F1':>16} | {'Δ Baru vs Lama':>14} | {'Δ Baru vs Gold':>14}"
)
print("-" * 92)
for row in EVAL_BUNDLE["per_class_rows"]:
    print(
        f"   {row['entity_class']:<5} | {row['support']:>7,} | {row['gold_f1_pct']:>7.2f}% | "
        f"{row['nemotron_18class_f1_pct']:>15.2f}% | {row['nemotron_old_bab4_f1_pct']:>15.2f}% | "
        f"{row['delta_new_vs_old_nemotron_f1_pct']:>+13.2f}% | {row['delta_new_nemotron_vs_gold_f1_pct']:>+13.2f}%"
    )
print("=" * 92)

In [ ]:
# =============================================================================
# SEL 10: ANALISIS GALAT OTOMATIS, VISUALISASI GRAFIK, PAKET .ZIP, & UPLOAD DRIVE OTOMATIS
# =============================================================================
from datetime import datetime, timezone
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def analyze_prediction_errors(
    records: list[dict],
    predictions: list[list[str]],
    entity_types: tuple[str, ...] = ACTIVE_TYPES,
) -> dict:
    """Klasifikasikan setiap galat prediksi pada Test Set ke dalam taksonomi galat CoNLL."""
    exact_tp = 0
    type_confusion_count = 0
    boundary_same_type_count = 0
    boundary_and_type_count = 0
    spurious_fp_count = 0
    missed_fn_count = 0

    confusion_pairs = Counter()
    spurious_by_class = Counter()
    missed_by_class = Counter()
    boundary_by_class = Counter()
    sample_errors = {
        "type_confusion": [],
        "boundary_error_same_type": [],
        "boundary_and_type_error": [],
        "spurious_fp": [],
        "missed_fn": [],
    }

    for rec, pred_tags in zip(records, predictions):
        tokens = rec["tokens"]
        gold_spans = spans(rec["tags"])
        pred_spans = spans(pred_tags)

        gold_set = set(gold_spans)
        pred_set = set(pred_spans)

        exact_matches = gold_set & pred_set
        exact_tp += len(exact_matches)

        rem_gold = [g for g in gold_spans if g not in exact_matches]
        rem_pred = [p for p in pred_spans if p not in exact_matches]

        matched_gold_idx = set()
        matched_pred_idx = set()

        # 1. Cek Type Confusion dengan batas rentang kata yang 100% sama persis
        gold_by_bounds = {(s, e): (idx, k) for idx, (s, e, k) in enumerate(rem_gold)}
        for p_idx, (ps, pe, pk) in enumerate(rem_pred):
            if (ps, pe) in gold_by_bounds:
                g_idx, gk = gold_by_bounds[(ps, pe)]
                if g_idx not in matched_gold_idx:
                    matched_gold_idx.add(g_idx)
                    matched_pred_idx.add(p_idx)
                    type_confusion_count += 1
                    confusion_pairs[(gk, pk)] += 1
                    if len(sample_errors["type_confusion"]) < 25:
                        sample_errors["type_confusion"].append({
                            "sentence_id": rec["sentence_id"],
                            "span_text": " ".join(tokens[ps:pe]),
                            "gold_type": gk,
                            "pred_type": pk,
                        })

        # 2. Cek Boundary Overlap (irisan token > 0 pada rentang yang belum terpasangkan)
        for p_idx, (ps, pe, pk) in enumerate(rem_pred):
            if p_idx in matched_pred_idx:
                continue
            best_g_idx = None
            best_overlap = 0
            for g_idx, (gs, ge, gk) in enumerate(rem_gold):
                if g_idx in matched_gold_idx:
                    continue
                overlap = max(0, min(pe, ge) - max(ps, gs))
                if overlap > best_overlap:
                    best_overlap = overlap
                    best_g_idx = g_idx
            if best_g_idx is not None and best_overlap > 0:
                gs, ge, gk = rem_gold[best_g_idx]
                matched_gold_idx.add(best_g_idx)
                matched_pred_idx.add(p_idx)
                if pk == gk:
                    boundary_same_type_count += 1
                    boundary_by_class[gk] += 1
                    if len(sample_errors["boundary_error_same_type"]) < 25:
                        sample_errors["boundary_error_same_type"].append({
                            "sentence_id": rec["sentence_id"],
                            "entity_type": gk,
                            "gold_text": " ".join(tokens[gs:ge]),
                            "pred_text": " ".join(tokens[ps:pe]),
                        })
                else:
                    boundary_and_type_count += 1
                    confusion_pairs[(gk, pk)] += 1
                    if len(sample_errors["boundary_and_type_error"]) < 25:
                        sample_errors["boundary_and_type_error"].append({
                            "sentence_id": rec["sentence_id"],
                            "gold_type": gk,
                            "gold_text": " ".join(tokens[gs:ge]),
                            "pred_type": pk,
                            "pred_text": " ".join(tokens[ps:pe]),
                        })

        # 3. Sisanya adalah Spurious False Positive & Missed False Negative murni
        for p_idx, (ps, pe, pk) in enumerate(rem_pred):
            if p_idx not in matched_pred_idx:
                spurious_fp_count += 1
                spurious_by_class[pk] += 1
                if len(sample_errors["spurious_fp"]) < 25:
                    sample_errors["spurious_fp"].append({
                        "sentence_id": rec["sentence_id"],
                        "pred_type": pk,
                        "pred_text": " ".join(tokens[ps:pe]),
                    })

        for g_idx, (gs, ge, gk) in enumerate(rem_gold):
            if g_idx not in matched_gold_idx:
                missed_fn_count += 1
                missed_by_class[gk] += 1
                if len(sample_errors["missed_fn"]) < 25:
                    sample_errors["missed_fn"].append({
                        "sentence_id": rec["sentence_id"],
                        "gold_type": gk,
                        "gold_text": " ".join(tokens[gs:ge]),
                    })

    total_non_tp_events = (
        type_confusion_count
        + boundary_same_type_count
        + boundary_and_type_count
        + spurious_fp_count
        + missed_fn_count
    )

    def pct(val: int) -> float:
        return round(100.0 * val / total_non_tp_events, 2) if total_non_tp_events else 0.0

    top_confusions = [
        {"gold_type": gk, "pred_type": pk, "count": cnt}
        for (gk, pk), cnt in confusion_pairs.most_common(30)
    ]

    return {
        "exact_match_tp": exact_tp,
        "total_error_events": total_non_tp_events,
        "counts": {
            "type_confusion": type_confusion_count,
            "boundary_error_same_type": boundary_same_type_count,
            "boundary_and_type_error": boundary_and_type_count,
            "spurious_fp": spurious_fp_count,
            "missed_fn": missed_fn_count,
        },
        "proportions_pct": {
            "type_confusion": pct(type_confusion_count),
            "boundary_error_same_type": pct(boundary_same_type_count),
            "boundary_and_type_error": pct(boundary_and_type_count),
            "spurious_fp": pct(spurious_fp_count),
            "missed_fn": pct(missed_fn_count),
        },
        "top_type_confusions": top_confusions,
        "spurious_fp_by_class": {k: spurious_by_class[k] for k in entity_types},
        "missed_fn_by_class": {k: missed_by_class[k] for k in entity_types},
        "boundary_error_by_class": {k: boundary_by_class[k] for k in entity_types},
        "sample_errors": sample_errors,
    }


def generate_experiment_figures(
    per_class_rows: list[dict],
    histories_dir: Path,
    error_breakdown: dict,
    figures_dir: Path = FIGURES_DIR,
    seed: int = TRAINING_SEEDS[0],
) -> list[Path]:
    """Buat grafik visualisasi perbandingan F1 per kelas, kurva konvergensi, dan komposisi galat."""
    figures_dir.mkdir(parents=True, exist_ok=True)
    generated = []

    # 1. Grafik Perbandingan F1 Per Kelas (18 Kelas)
    classes = [r["entity_class"] for r in per_class_rows]
    gold_f1s = [r["gold_f1_pct"] for r in per_class_rows]
    nem_new_f1s = [r["nemotron_18class_f1_pct"] for r in per_class_rows]
    nem_old_f1s = [r["nemotron_old_bab4_f1_pct"] for r in per_class_rows]

    fig, ax = plt.subplots(figsize=(15, 6.5), dpi=300)
    x_pos = list(range(len(classes)))
    width = 0.27

    ax.bar([x - width for x in x_pos], gold_f1s, width=width, label="NERGRIT Gold Baseline", color="#1f77b4")
    ax.bar(x_pos, nem_new_f1s, width=width, label="Nemotron 18-Kelas Baru (Terverifikasi)", color="#2ca02c")
    ax.bar([x + width for x in x_pos], nem_old_f1s, width=width, label="Nemotron Lama (Bab 4 Sebelum Verifikasi)", color="#ff7f0e", alpha=0.85)

    ax.set_title("Perbandingan Skor F1 per Kelas Entitas pada NERGRIT Gold Test Set (18 Kelas)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Kelas Entitas (18 Kelas Standar BIO)", fontsize=11)
    ax.set_ylabel("Exact Span F1-Score (%)", fontsize=11)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(classes, fontsize=10, fontweight="bold")
    ax.set_ylim(0, 108)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.legend(loc="upper right", fontsize=10)
    fig.tight_layout()

    f1_chart_path = figures_dir / "per_class_f1_comparison.png"
    fig.savefig(f1_chart_path, dpi=300)
    plt.close(fig)
    generated.append(f1_chart_path)

    # 2. Grafik Kurva Konvergensi Pelatihan (Loss & Validation Macro-F1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), dpi=300)
    for prefix, title_label, color in (
        ("gold", "NERGRIT Gold", "#1f77b4"),
        ("nemotron", "Nemotron 18-Kelas Baru", "#2ca02c"),
    ):
        hist_json = histories_dir / f"{prefix}_s{seed}.json"
        if not hist_json.is_file():
            continue
        history = json.loads(hist_json.read_text(encoding="utf-8"))
        eval_rows = [r for r in history if "eval_loss" in r and "epoch" in r]
        train_rows = [r for r in history if "loss" in r and "eval_loss" not in r and "epoch" in r]

        if train_rows:
            axes[0].plot(
                [r["epoch"] for r in train_rows],
                [r["loss"] for r in train_rows],
                linestyle="--",
                marker="o",
                color=color,
                alpha=0.75,
                label=f"{title_label} — Train Loss",
            )
        if eval_rows:
            axes[0].plot(
                [r["epoch"] for r in eval_rows],
                [r["eval_loss"] for r in eval_rows],
                linestyle="-",
                marker="s",
                color=color,
                label=f"{title_label} — Val Loss",
            )
            axes[1].plot(
                [r["epoch"] for r in eval_rows],
                [r.get("eval_macro_f1", 0.0) * 100.0 for r in eval_rows],
                linestyle="-",
                marker="o",
                color=color,
                linewidth=2.0,
                label=f"{title_label} — Val Macro-F1",
            )
            axes[1].plot(
                [r["epoch"] for r in eval_rows],
                [r.get("eval_micro_f1", 0.0) * 100.0 for r in eval_rows],
                linestyle=":",
                marker="^",
                color=color,
                alpha=0.8,
                label=f"{title_label} — Val Micro-F1",
            )

    axes[0].set_title("Dinamika Training & Validation Loss per Epoch", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Cross-Entropy Loss")
    axes[0].grid(True, linestyle="--", alpha=0.4)
    axes[0].legend(fontsize=9)

    axes[1].set_title("Konvergensi Validation Macro-F1 & Micro-F1 per Epoch", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("F1-Score (%)")
    axes[1].grid(True, linestyle="--", alpha=0.4)
    axes[1].legend(fontsize=9)
    fig.tight_layout()

    conv_chart_path = figures_dir / "training_convergence_curves.png"
    fig.savefig(conv_chart_path, dpi=300)
    plt.close(fig)
    generated.append(conv_chart_path)

    # 3. Grafik Komposisi Taksonomi Galat (Error Breakdown)
    err_categories = [
        ("type_confusion", "Type Confusion\n(Batas Tepat)"),
        ("boundary_error_same_type", "Boundary Error\n(Kelas Tepat)"),
        ("boundary_and_type_error", "Boundary +\nType Error"),
        ("spurious_fp", "Spurious FP\n(Over-Tagging)"),
        ("missed_fn", "Missed FN\n(Entitas Terlewat)"),
    ]
    g_err = error_breakdown["gold"]["counts"]
    n_err = error_breakdown["nemotron_18class"]["counts"]

    fig, ax = plt.subplots(figsize=(10, 5.2), dpi=300)
    idx_pos = list(range(len(err_categories)))
    w = 0.35
    ax.bar(
        [i - w / 2 for i in idx_pos],
        [g_err[k] for k, _ in err_categories],
        width=w,
        label="NERGRIT Gold Baseline",
        color="#1f77b4",
    )
    ax.bar(
        [i + w / 2 for i in idx_pos],
        [n_err[k] for k, _ in err_categories],
        width=w,
        label="Nemotron 18-Kelas Baru",
        color="#2ca02c",
    )
    ax.set_title("Perbandingan Distribusi Kategori Galat pada NERGRIT Gold Test Set", fontsize=12, fontweight="bold")
    ax.set_xticks(idx_pos)
    ax.set_xticklabels([lbl for _, lbl in err_categories], fontsize=10)
    ax.set_ylabel("Jumlah Kejadian Galat (Rentang Entitas)")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.legend(fontsize=10)
    fig.tight_layout()

    err_chart_path = figures_dir / "error_taxonomy_breakdown.png"
    fig.savefig(err_chart_path, dpi=300)
    plt.close(fig)
    generated.append(err_chart_path)

    for fig_p in generated:
        sync_to_drive(fig_p, f"figures/{fig_p.name}")

    return generated


def package_experiment_results_zip(
    output_dir: Path = OUTPUT_DIR,
    zip_filename: str = "nemotron_18class_experiment_results.zip",
) -> tuple[Path, dict]:
    """Kemas seluruh hasil eksperimen ke dalam arsip .zip dan unggah otomatis ke Google Drive."""
    zip_path = output_dir / zip_filename
    folders_to_include = ("tables", "histories", "predictions", "error_analysis", "figures")
    root_files_to_include = (
        "experiment_summary.json",
        "experiment_config.json",
        "environment_fingerprint.json",
        "bundle_verification.json",
    )

    packaged_files = {}
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fname in root_files_to_include:
            fpath = output_dir / fname
            if fpath.is_file():
                zf.write(fpath, arcname=fname)
                packaged_files[fname] = compute_sha256(fpath)

        for folder in folders_to_include:
            folder_path = output_dir / folder
            if not folder_path.is_dir():
                continue
            for fpath in sorted(folder_path.rglob("*")):
                if fpath.is_file():
                    rel_arc = f"{folder}/{fpath.relative_to(folder_path).as_posix()}"
                    zf.write(fpath, arcname=rel_arc)
                    packaged_files[rel_arc] = compute_sha256(fpath)

        manifest_payload = {
            "archive_name": zip_filename,
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "file_count": len(packaged_files),
            "files_sha256": packaged_files,
        }
        zf.writestr("results_manifest.json", json.dumps(manifest_payload, indent=2, ensure_ascii=False))

    # Salin juga ke working directory root (/content jika di Colab) agar mudah diakses
    if IN_COLAB:
        colab_root_zip = Path("/content") / zip_filename
        if colab_root_zip.resolve() != zip_path.resolve():
            shutil.copy2(zip_path, colab_root_zip)

    # Unggah otomatis berkas .zip ke Google Drive dan verifikasi SHA-256
    drive_zip = sync_to_drive(zip_path, zip_filename, verify_hash=True)
    return zip_path, {
        "zip_path": str(zip_path),
        "drive_zip_path": str(drive_zip) if drive_zip else None,
        "zip_sha256": compute_sha256(zip_path),
        "zip_size_bytes": zip_path.stat().st_size,
        "file_count": len(packaged_files),
    }


# 1. Jalankan Analisis Galat Otomatis pada Gold Test Set
_primary_seed = TRAINING_SEEDS[0]
ERROR_BREAKDOWN = {
    "seed": _primary_seed,
    "gold": analyze_prediction_errors(
        PREPARED_DATA["gold"]["test"],
        EVAL_BUNDLE["predictions_by_run"]["gold"][_primary_seed],
        ACTIVE_TYPES,
    ),
    "nemotron_18class": analyze_prediction_errors(
        PREPARED_DATA["gold"]["test"],
        EVAL_BUNDLE["predictions_by_run"]["nemotron_18class"][_primary_seed],
        ACTIVE_TYPES,
    ),
}

error_json_path = ERROR_ANALYSIS_DIR / "error_breakdown.json"
error_json_path.write_text(json.dumps(ERROR_BREAKDOWN, indent=2, ensure_ascii=False), encoding="utf-8")
sync_to_drive(error_json_path, "error_analysis/error_breakdown.json")

error_table_rows = []
for sc_key, sc_label in (("gold", "NERGRIT Gold Baseline"), ("nemotron_18class", "Nemotron 18-Kelas Baru")):
    eb = ERROR_BREAKDOWN[sc_key]
    error_table_rows.append({
        "scenario_key": sc_key,
        "scenario_name": sc_label,
        "seed": _primary_seed,
        "exact_match_tp": eb["exact_match_tp"],
        "total_error_events": eb["total_error_events"],
        "type_confusion_count": eb["counts"]["type_confusion"],
        "type_confusion_pct": eb["proportions_pct"]["type_confusion"],
        "boundary_same_type_count": eb["counts"]["boundary_error_same_type"],
        "boundary_same_type_pct": eb["proportions_pct"]["boundary_error_same_type"],
        "boundary_and_type_count": eb["counts"]["boundary_and_type_error"],
        "boundary_and_type_pct": eb["proportions_pct"]["boundary_and_type_error"],
        "spurious_fp_count": eb["counts"]["spurious_fp"],
        "spurious_fp_pct": eb["proportions_pct"]["spurious_fp"],
        "missed_fn_count": eb["counts"]["missed_fn"],
        "missed_fn_pct": eb["proportions_pct"]["missed_fn"],
    })
write_csv_rows(TABLES_DIR / "error_taxonomy_comparison.csv", error_table_rows)
sync_to_drive(TABLES_DIR / "error_taxonomy_comparison.csv", "tables/error_taxonomy_comparison.csv")

# 2. Buat Grafik Visualisasi (.png)
GENERATED_FIGURES = generate_experiment_figures(
    per_class_rows=EVAL_BUNDLE["per_class_rows"],
    histories_dir=HISTORIES_DIR,
    error_breakdown=ERROR_BREAKDOWN,
    figures_dir=FIGURES_DIR,
    seed=_primary_seed,
)

# 3. Susun Ringkasan Eksekutif Lengkap (experiment_summary.json)
EXPERIMENT_SUMMARY = {
    "status": "complete",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "configuration": CONFIG,
    "environment": ENV_FINGERPRINT,
    "dataset_statistics": PREPARED_MANIFEST["statistics"],
    "leakage_verification": PREPARED_MANIFEST["leakage_check"],
    "training_runs": {
        "gold": {str(k): v for k, v in GOLD_RUN_MANIFESTS.items()},
        "nemotron_18class": {str(k): v for k, v in NEMOTRON_RUN_MANIFESTS.items()},
    },
    "evaluation_gold_test": {
        "overall_rows": EVAL_BUNDLE["overall_rows"],
        "delta_nemotron_18class_vs_old_nemotron": EVAL_BUNDLE["delta_vs_old"],
        "delta_nemotron_18class_vs_gold": EVAL_BUNDLE["delta_vs_gold"],
        "per_class_18_comparison": EVAL_BUNDLE["per_class_rows"],
    },
    "error_analysis_summary": {
        "gold": {
            "exact_match_tp": ERROR_BREAKDOWN["gold"]["exact_match_tp"],
            "counts": ERROR_BREAKDOWN["gold"]["counts"],
            "proportions_pct": ERROR_BREAKDOWN["gold"]["proportions_pct"],
            "top_type_confusions": ERROR_BREAKDOWN["gold"]["top_type_confusions"][:15],
        },
        "nemotron_18class": {
            "exact_match_tp": ERROR_BREAKDOWN["nemotron_18class"]["exact_match_tp"],
            "counts": ERROR_BREAKDOWN["nemotron_18class"]["counts"],
            "proportions_pct": ERROR_BREAKDOWN["nemotron_18class"]["proportions_pct"],
            "top_type_confusions": ERROR_BREAKDOWN["nemotron_18class"]["top_type_confusions"][:15],
        },
    },
}

summary_path = OUTPUT_DIR / "experiment_summary.json"
summary_path.write_text(json.dumps(EXPERIMENT_SUMMARY, indent=2, ensure_ascii=False), encoding="utf-8")
sync_to_drive(summary_path, "experiment_summary.json")

# 4. Kemas Seluruh Artefak ke nemotron_18class_experiment_results.zip & Upload Otomatis ke Google Drive
RESULTS_ZIP_PATH, ZIP_INFO = package_experiment_results_zip(OUTPUT_DIR)

print("\n" + "=" * 88)
print("🎉 EKSPERIMEN SELESAI & ARSIP .ZIP BERHASIL DISIMPAN KE GOOGLE DRIVE!")
print("=" * 88)
print(f"   Berkas ZIP Lokal       : {ZIP_INFO['zip_path']} ({ZIP_INFO['zip_size_bytes'] / (1024 * 1024):.2f} MB)")
print(f"   Berkas ZIP Google Drive: {ZIP_INFO['drive_zip_path']}")
print(f"   SHA-256 Arsip ZIP      : {ZIP_INFO['zip_sha256']}")
print(f"   Jumlah Berkas di ZIP   : {ZIP_INFO['file_count']} berkas")
print("=" * 88)

try:
    from IPython.display import FileLink, display  # type: ignore
    display(FileLink(str(RESULTS_ZIP_PATH)))
except Exception:
    pass

if IN_COLAB:
    try:
        from google.colab import files  # type: ignore
        print("⬇️ Memulai pengunduhan otomatis nemotron_18class_experiment_results.zip ke peramban...")
        files.download(str(RESULTS_ZIP_PATH))
    except Exception as exc:
        print(f"ℹ️ Unduhan langsung peramban dilewati ({exc}). Berkas sudah aman di Google Drive: {ZIP_INFO['drive_zip_path']}")